# Classificação por Quadrantes Emocionais — Fingerprints vs openSMILE Baseline

Este notebook avalia as **features de fingerprinting emocional** na tarefa de classificação dos
quatro quadrantes emocionais do espaço Valence–Arousal, comparando diretamente com o **baseline
openSMILE** estabelecido em `PyCaret_Quadrantes_Emocionais_openSMILE.ipynb`.

**Fluxo:**
1. Carregamento do dataset de fingerprints por bloco (`fingerprint_features_by_block.parquet`).
2. Re-derivação dos rótulos de quadrante com o mesmo limiar do baseline (zero na escala –1…1).
3. Inventário e ranking ANOVA das features de fingerprint.
4. Validação com `GroupKFold` por `song_id` — mesma configuração do baseline.
5. Comparação direta dos resultados fingerprint vs openSMILE.
6. Análise por fold, matriz de confusão e resumo para o TCC.
7. Comparacao final das tecnicas de fingerprint (`rank` vs `band-rank`) cruzadas com a origem (`bloco` vs `raw-peaks`).


## 1. Imports

In [1]:
import os
import re
import warnings
from pathlib import Path

# Avoid noisy joblib/loky physical-core detection tracebacks on Windows.
os.environ.setdefault("LOKY_MAX_CPU_COUNT", str(os.cpu_count() or 1))
try:
    N_JOBS_MODEL = max(1, int(os.environ["LOKY_MAX_CPU_COUNT"]))
except ValueError:
    N_JOBS_MODEL = max(1, os.cpu_count() or 1)
    os.environ["LOKY_MAX_CPU_COUNT"] = str(N_JOBS_MODEL)

warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", message="Could not find the number of physical cores.*")

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from tqdm import tqdm
from IPython.display import display, Markdown

from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    GradientBoostingClassifier,
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import GroupKFold
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from scipy.stats import ttest_rel, wilcoxon

try:
    from sklearn.model_selection import StratifiedGroupKFold
except ImportError:
    StratifiedGroupKFold = None

print(f"pandas {pd.__version__}  |  numpy {np.__version__}  |  n_jobs_model={N_JOBS_MODEL}")

pandas 2.1.4  |  numpy 1.26.4  |  n_jobs_model=12


## 2. Configuração

In [2]:
# ===========================
# CAMINHOS
# ===========================
PROJECT_DIR = Path(r"C:\dev\python\TCC Ajustado")
DATA_DIR    = PROJECT_DIR / "Dados"

FINGERPRINT_BLOCK_PATH = DATA_DIR / "fingerprint_band_rank" / "fingerprint_band_rank.parquet"
FINGERPRINT_BAND_RANK_RAW_PEAKS_PATH = DATA_DIR / "fingerprint_band_rank" / "fingerprint_band_rank_raw_peaks.parquet"
FINGERPRINT_RANK_BLOCK_PATH = DATA_DIR / "fingerprint_rank" / "fingerprint_rank.parquet"

# Baseline openSMILE: resultados já salvos pelo notebook de quadrantes openSMILE
BASELINE_TABLES_PATH = DATA_DIR / "pycaret_quadrants_outputs" / "tables"

OUTPUT_PATH   = DATA_DIR / "pycaret_quadrants_fingerprints_outputs"
FIGURES_PATH  = OUTPUT_PATH / "figures"
TABLES_PATH   = OUTPUT_PATH / "tables"
MODELS_PATH   = OUTPUT_PATH / "models"
for p in [FIGURES_PATH, TABLES_PATH, MODELS_PATH]:
    p.mkdir(parents=True, exist_ok=True)

# ===========================
# COLUNAS
# ===========================
SONG_ID_COL   = "song_id"
BLOCK_ID_COL  = "block_id"
START_COL     = "block_start_sec"
END_COL       = "block_end_sec"
VALENCE_COL   = "valence"
AROUSAL_COL   = "arousal"
QUADRANT_COL  = "quadrant_label_std"   # coluna re-derivada neste notebook

META_COLS = {
    SONG_ID_COL, BLOCK_ID_COL, START_COL, END_COL,
    "block_duration_sec", "valence", "arousal",
    "quadrant", "quadrant_label", "quadrant_label_std",
    "method", "title", "artist", "genre", "filename",
    # band-rank raw-peaks metadata columns
    "band_name", "band_id", "band_low_hz", "band_high_hz", "topk_per_band",
    # legacy / fallback names
    "block_idx", "start_time", "end_time", "duration_sec",
}

# ===========================
# QUADRANTES — mesmo protocolo do baseline openSMILE
# ===========================
# Escala VA: –1 a 1  →  limiar = 0
VALENCE_THRESHOLD = 0.0
AROUSAL_THRESHOLD = 0.0
VA_SCALE = "-1_to_1"

QUADRANT_ORDER = [
    "Q1_Alegre_Energetico",
    "Q2_Tenso_Raivoso",
    "Q3_Triste_Melancolico",
    "Q4_Calmo_Relaxado",
]
QUADRANT_COLOR_MAP = {
    "Q1_Alegre_Energetico":  "#2E7D32",
    "Q2_Tenso_Raivoso":      "#C62828",
    "Q3_Triste_Melancolico": "#1565C0",
    "Q4_Calmo_Relaxado":     "#F9A825",
}
QUADRANT_SHORT = {
    "Q1_Alegre_Energetico":  "Q1 Alegre/Energ.",
    "Q2_Tenso_Raivoso":      "Q2 Tenso/Raivoso",
    "Q3_Triste_Melancolico": "Q3 Triste/Melanc.",
    "Q4_Calmo_Relaxado":     "Q4 Calmo/Relaxado",
}

# ===========================
# MODELAGEM
# ===========================
RANDOM_STATE      = 42
N_SPLITS          = 5
SELECTOR_K_VALUES = [30, 60, 100, 200]
MAX_MISSING_RATE  = 0.40
MAX_SAMPLE_FOR_EDA = 50_000

# ===========================
# GRÁFICOS
# ===========================
EXPORT_PNG      = True
EXPORT_SVG      = True
FIG_WIDTH       = 1200
FIG_HEIGHT      = 720
FIG_SCALE       = 2
PLOT_TEMPLATE   = "plotly_white"
SHOW_FIGURES    = True

print("Saída:", OUTPUT_PATH)
print("Baseline openSMILE:", BASELINE_TABLES_PATH)

Saída: C:\dev\python\TCC Ajustado\Dados\pycaret_quadrants_fingerprints_outputs
Baseline openSMILE: C:\dev\python\TCC Ajustado\Dados\pycaret_quadrants_outputs\tables


## 3. Funções utilitárias

In [3]:
def save_table(df_out, filename):
    path = TABLES_PATH / filename
    df_out.to_csv(path, index=False, encoding="utf-8-sig")
    print(f"Tabela salva: {path}")
    return path


def save_fig(fig, name):
    fig.update_layout(template=PLOT_TEMPLATE, width=FIG_WIDTH, height=FIG_HEIGHT)
    if SHOW_FIGURES:
        fig.show()
    if EXPORT_PNG:
        try:
            path = FIGURES_PATH / f"{name}.png"
            fig.write_image(str(path), scale=FIG_SCALE)
        except Exception as e:
            print(f"[AVISO] PNG não exportado: {e}")
    if EXPORT_SVG:
        try:
            path = FIGURES_PATH / f"{name}.svg"
            fig.write_image(str(path))
        except Exception as e:
            print(f"[AVISO] SVG não exportado: {e}")


def safe_sample(df_in, max_rows=MAX_SAMPLE_FOR_EDA, random_state=RANDOM_STATE):
    if len(df_in) <= max_rows:
        return df_in.copy()
    return df_in.sample(max_rows, random_state=random_state).copy()


def numeric_feature_cols(df, exclude=None):
    exclude = set(exclude or [])
    result = []
    for col in df.columns:
        if col in exclude:
            continue
        s = pd.to_numeric(df[col], errors="coerce")
        if s.notna().sum() == 0 or s.nunique(dropna=True) <= 1:
            continue
        result.append(col)
    return result


def feature_missing_rate(df, cols):
    return df[cols].isnull().mean()


print("Utilitários carregados.")

Utilitários carregados.


In [4]:
def normalize_key_columns(df):
    out = df.copy()
    lower_to_original = {str(col).strip().lower(): col for col in out.columns}
    rename_map = {}

    if SONG_ID_COL not in out.columns:
        for alias in ["song_id", "songid", "song id", "track_id", "trackid", "track id"]:
            if alias in lower_to_original:
                rename_map[lower_to_original[alias]] = SONG_ID_COL
                break

    if BLOCK_ID_COL not in out.columns:
        for alias in ["block_idx", "blockid", "block id", "segment_idx", "segmentid", "segment id", "window_idx", "window id"]:
            if alias in lower_to_original:
                rename_map[lower_to_original[alias]] = BLOCK_ID_COL
                break

    if rename_map:
        out = out.rename(columns=rename_map)
    return out



def is_leakage_like_column(col):
    name = str(col).strip().lower()
    leakage_tokens = [
        "quadrant", "valence", "arousal", "label", "target", "class",
        "song_id", "block_id", "block_idx", "artist", "genre", "title", "method",
        "duration", "block_start", "block_end", "start_time", "end_time",
        "filepath", "file_path", "filename", "source",
    ]
    return any(token in name for token in leakage_tokens)



def make_leakage_report(feature_cols, name="features"):
    cols = [str(col) for col in feature_cols]
    suspect_cols = [col for col in cols if is_leakage_like_column(col)]
    status = "ok" if not suspect_cols else "suspect"
    return pd.DataFrame([{
        "feature_set": name,
        "status": status,
        "n_features": len(cols),
        "n_suspect": len(suspect_cols),
        "suspect_columns": ", ".join(suspect_cols[:30]),
    }])



def _extract_numeric_feature_cols(df):
    candidate_cols = []
    for col in df.columns:
        if col in META_COLS or is_leakage_like_column(col):
            continue
        series = pd.to_numeric(df[col], errors="coerce")
        if series.notna().sum() == 0:
            continue
        if series.nunique(dropna=True) <= 1:
            continue
        candidate_cols.append(col)
    return candidate_cols



def load_opensmile_features_for_fusion():
    search_roots = [PROJECT_DIR, DATA_DIR, BASELINE_TABLES_PATH]
    candidate_names = [
        "quadrant_modeling_dataset.parquet",
        "quadrant_modeling_dataset.csv",
        "openSMILE_quadrant_modeling_dataset.parquet",
        "openSMILE_quadrant_modeling_dataset.csv",
        "opensmile_quadrant_modeling_dataset.parquet",
        "opensmile_quadrant_modeling_dataset.csv",
        "opensmile_features_by_block.parquet",
        "opensmile_features_by_block.csv",
    ]
    candidate_patterns = [
        "*opensmile*.parquet",
        "*opensmile*.csv",
        "*openSMILE*.parquet",
        "*openSMILE*.csv",
        "*quadrant*modeling*dataset*.parquet",
        "*quadrant*modeling*dataset*.csv",
    ]

    discovered_paths = []
    for root in search_roots:
        if root is None or not Path(root).exists():
            continue
        for name in candidate_names:
            discovered_paths.extend(Path(root).rglob(name))
        for pattern in candidate_patterns:
            discovered_paths.extend(Path(root).rglob(pattern))

    unique_paths = []
    seen = set()
    for path in discovered_paths:
        resolved = Path(path)
        key = str(resolved).lower()
        if key in seen:
            continue
        seen.add(key)
        unique_paths.append(resolved)

    audit_rows = []
    best_df = pd.DataFrame()
    best_feature_cols = []
    best_path = None
    best_score = -1

    for path in unique_paths:
        try:
            if path.suffix.lower() == ".parquet":
                candidate_df = pd.read_parquet(path)
            elif path.suffix.lower() == ".csv":
                candidate_df = pd.read_csv(path, encoding="utf-8-sig")
            else:
                continue

            candidate_df.columns = [str(col).strip() for col in candidate_df.columns]
            candidate_df = normalize_key_columns(candidate_df)
            feature_cols = _extract_numeric_feature_cols(candidate_df)
            score = len(feature_cols)
            audit_rows.append({
                "path": str(path),
                "exists": True,
                "rows": int(candidate_df.shape[0]),
                "cols": int(candidate_df.shape[1]),
                "feature_cols": int(len(feature_cols)),
                "has_song_id": SONG_ID_COL in candidate_df.columns,
                "has_block_id": BLOCK_ID_COL in candidate_df.columns,
            })
            if score > best_score and SONG_ID_COL in candidate_df.columns and BLOCK_ID_COL in candidate_df.columns:
                best_score = score
                best_df = candidate_df.copy()
                best_feature_cols = feature_cols
                best_path = path
        except Exception as exc:
            audit_rows.append({
                "path": str(path),
                "exists": True,
                "rows": np.nan,
                "cols": np.nan,
                "feature_cols": np.nan,
                "has_song_id": False,
                "has_block_id": False,
                "error": str(exc),
            })

    audit_df = pd.DataFrame(audit_rows)
    if best_df.empty:
        return pd.DataFrame(), [], None, audit_df
    return best_df, best_feature_cols, best_path, audit_df



def align_cv_for_dataset(df, group_col=SONG_ID_COL):
    n_groups = df[group_col].astype(str).nunique()
    n_splits = min(N_SPLITS, n_groups)
    if n_splits < 2:
        raise RuntimeError("São necessários pelo menos 2 grupos para validação cruzada.")
    if StratifiedGroupKFold is not None and QUADRANT_COL in df.columns and df[QUADRANT_COL].nunique() > 1:
        return StratifiedGroupKFold(n_splits=n_splits), n_splits
    return GroupKFold(n_splits=n_splits), n_splits



def best_non_dummy(results_df):
    if results_df is None or results_df.empty:
        return pd.Series(dtype=object)
    ok = results_df.copy()
    if "status" in ok.columns:
        ok = ok[ok["status"].eq("ok")].copy()
    if ok.empty or "model" not in ok.columns or "macro_f1_mean" not in ok.columns:
        return pd.Series(dtype=object)
    ok = ok[~ok["model"].astype(str).str.startswith("Dummy")].copy()
    if ok.empty:
        return pd.Series(dtype=object)
    sort_cols = [c for c in ["macro_f1_mean", "balanced_accuracy_mean", "accuracy_mean"] if c in ok.columns]
    return ok.sort_values(sort_cols, ascending=[False] * len(sort_cols), na_position="last").iloc[0]


RUN_FUSION_VARIANTS = True
FUSION_TOP_K_VALUES = [20, 40, 80]
FUSION_BANDS = ["low", "mid", "high", "very_high", "global"]
FUSION_METRICS = ["magnitude", "rank", "pitch_octave", "counts", "dispersion", "central_tendency", "frequency", "other"]

print("Helpers de fusão carregados.")

Helpers de fusão carregados.


## 4. Carregamento e preparação dos dados

O dataset de fingerprints já contém `valence` e `arousal` alinhados por bloco de 10 s.
Os rótulos de quadrante são **re-derivados** aqui com o mesmo limiar do baseline openSMILE
(zero na escala –1…1) para garantir comparabilidade direta.

In [5]:
_band_df = pd.read_parquet(FINGERPRINT_BLOCK_PATH)
_band_df.columns = [str(c).strip() for c in _band_df.columns]

_rank_df = pd.read_parquet(FINGERPRINT_RANK_BLOCK_PATH)
_rank_df.columns = [str(c).strip() for c in _rank_df.columns]

_MERGE_KEYS = [SONG_ID_COL, BLOCK_ID_COL, START_COL, END_COL]


def _slug_feature_part(value):
    text = str(value).strip().lower()
    text = re.sub(r"[^0-9a-zA-Z]+", "_", text)
    return text.strip("_") or "unknown"


def _aggregate_peak_rows_by_block(peaks_df, prefix):
    """Agrega uma tabela granular de picos em features por bloco."""
    if peaks_df is None or peaks_df.empty:
        return pd.DataFrame(columns=_MERGE_KEYS)

    peaks = peaks_df.copy()
    peaks.columns = [str(c).strip() for c in peaks.columns]
    peaks = peaks.dropna(subset=[c for c in _MERGE_KEYS if c in peaks.columns]).copy()
    if peaks.empty:
        return pd.DataFrame(columns=_MERGE_KEYS)

    numeric_cols = [
        "frequency_hz", "magnitude", "magnitude_db", "magnitude_norm",
        "peak_time_relative_sec", "peak_time_sec", "midi_note", "octave",
        "semitone", "peak_rank_in_band", "peak_rank_global", "frame_idx", "freq_bin",
    ]
    numeric_cols = [c for c in numeric_cols if c in peaks.columns]
    for col in numeric_cols:
        peaks[col] = pd.to_numeric(peaks[col], errors="coerce")

    def _make_agg(block, local_prefix):
        parts = []
        if numeric_cols:
            agg = block.groupby(_MERGE_KEYS)[numeric_cols].agg(["mean", "std", "min", "max"])
            agg.columns = [f"{local_prefix}_{col}_{stat}" for col, stat in agg.columns]
            parts.append(agg)

        counts = block.groupby(_MERGE_KEYS).size().to_frame(f"{local_prefix}_peak_count")
        parts.append(counts)

        unique_cols = [c for c in ["freq_bin", "midi_note", "pitch_class", "frame_idx"] if c in block.columns]
        for col in unique_cols:
            nunique = block.groupby(_MERGE_KEYS)[col].nunique(dropna=True).to_frame(
                f"{local_prefix}_{col}_nunique"
            )
            parts.append(nunique)

        return pd.concat(parts, axis=1).reset_index()

    out = _make_agg(peaks, f"{prefix}_global")

    if "band_name" in peaks.columns:
        for band_name, band_block in peaks.groupby("band_name"):
            band_prefix = f"{prefix}_{_slug_feature_part(band_name)}"
            band_agg = _make_agg(band_block, band_prefix)
            out = out.merge(band_agg, on=_MERGE_KEYS, how="left")

    return out


if FINGERPRINT_BAND_RANK_RAW_PEAKS_PATH.exists():
    _band_raw_peaks_df = pd.read_parquet(FINGERPRINT_BAND_RANK_RAW_PEAKS_PATH)
    _band_raw_peaks_block_df = _aggregate_peak_rows_by_block(_band_raw_peaks_df, "raw")
else:
    _band_raw_peaks_df = pd.DataFrame()
    _band_raw_peaks_block_df = pd.DataFrame(columns=_MERGE_KEYS)

# Identify feature columns (non-meta) in each file and prefix them.
_band_feat = [c for c in _band_df.columns if c not in META_COLS]
_rank_feat = [c for c in _rank_df.columns if c not in META_COLS]
_band_raw_feat = [c for c in _band_raw_peaks_block_df.columns if c not in _MERGE_KEYS]

_band_df = _band_df.rename(columns={c: f"fpband__{c}" for c in _band_feat})
_rank_df = _rank_df.rename(columns={c: f"fprank__{c}" for c in _rank_feat})
_band_raw_peaks_block_df = _band_raw_peaks_block_df.rename(
    columns={c: f"fpbandraw__{c}" for c in _band_raw_feat}
)

_rank_prefixed = [f"fprank__{c}" for c in _rank_feat]
_band_raw_prefixed = [f"fpbandraw__{c}" for c in _band_raw_feat]

raw_df = _band_df.merge(
    _rank_df[_MERGE_KEYS + _rank_prefixed],
    on=_MERGE_KEYS,
    how="inner",
)

if _band_raw_prefixed:
    raw_df = raw_df.merge(
        _band_raw_peaks_block_df[_MERGE_KEYS + _band_raw_prefixed],
        on=_MERGE_KEYS,
        how="left",
    )

print(f"Band Rank: {_band_df.shape[0]:,} blocos, {len(_band_feat)} features")
print(f"Band Rank raw-peaks: {_band_raw_peaks_df.shape[0]:,} picos, {len(_band_raw_feat)} features agregadas por bloco")
print(f"Rank: {_rank_df.shape[0]:,} blocos, {len(_rank_feat)} features")
print(f"Dataset merged: {raw_df.shape[0]:,} linhas x {raw_df.shape[1]} colunas")
print("Colunas de metadados presentes:", [c for c in raw_df.columns if c in META_COLS])


Band Rank: 6,506 blocos, 347 features
Band Rank raw-peaks: 260,240 picos, 285 features agregadas por bloco
Rank: 6,506 blocos, 208 features
Dataset merged: 6,506 linhas x 851 colunas
Colunas de metadados presentes: ['song_id', 'filename', 'block_id', 'block_start_sec', 'block_end_sec', 'block_duration_sec', 'valence', 'arousal', 'quadrant', 'quadrant_label', 'method']


In [6]:
def add_standardized_quadrant(df, v_thresh=VALENCE_THRESHOLD, a_thresh=AROUSAL_THRESHOLD):
    """Re-deriva quadrant_label_std com os mesmos limiares do baseline openSMILE."""
    out  = df.copy()
    v    = pd.to_numeric(out[VALENCE_COL], errors="coerce")
    a    = pd.to_numeric(out[AROUSAL_COL], errors="coerce")
    high_v = v >= v_thresh
    high_a = a >= a_thresh

    conditions = [
        high_v & high_a,
        ~high_v & high_a,
        ~high_v & ~high_a,
        high_v & ~high_a,
    ]
    labels = [
        "Q1_Alegre_Energetico",
        "Q2_Tenso_Raivoso",
        "Q3_Triste_Melancolico",
        "Q4_Calmo_Relaxado",
    ]
    out[QUADRANT_COL] = np.select(conditions, labels, default=np.nan)
    # remove linhas sem VA válido
    out = out.dropna(subset=[QUADRANT_COL]).copy()
    return out


df = add_standardized_quadrant(raw_df)

print(f"Após filtro VA: {df.shape[0]:,} blocos")
print("Distribuição dos quadrantes:")
print(df[QUADRANT_COL].value_counts())

Após filtro VA: 6,506 blocos
Distribuição dos quadrantes:
quadrant_label_std
Q1_Alegre_Energetico     3217
Q3_Triste_Melancolico    1382
Q2_Tenso_Raivoso         1019
Q4_Calmo_Relaxado         888
Name: count, dtype: int64


In [7]:
# Identifica colunas de features de fingerprint
all_feature_cols = [c for c in df.columns if c not in META_COLS]
all_feature_cols = [c for c in all_feature_cols if c not in [
    SONG_ID_COL, BLOCK_ID_COL, START_COL, END_COL,
    VALENCE_COL, AROUSAL_COL, "block_duration_sec",
    "quadrant", "quadrant_label", "method", "title", "artist", "genre",
    QUADRANT_COL,
]]

# Remove features com missing excessivo ou sem variância
missing_rate = feature_missing_rate(df, all_feature_cols)
usable_feature_cols = [
    c for c in all_feature_cols
    if missing_rate[c] <= MAX_MISSING_RATE
    and pd.to_numeric(df[c], errors="coerce").nunique(dropna=True) > 1
]

print(f"Features totais: {len(all_feature_cols)}")
print(f"Features utilizáveis (missing ≤ {MAX_MISSING_RATE:.0%}, variância > 0): {len(usable_feature_cols)}")

Features totais: 840
Features utilizáveis (missing ≤ 40%, variância > 0): 755


## 5. EDA — distribuição de classes e espaço VA

In [8]:
class_balance = (
    df[QUADRANT_COL]
    .value_counts()
    .reset_index()
    .rename(columns={QUADRANT_COL: "count", "index": QUADRANT_COL})
)
# compatível com pandas ≥ 2.0
if "count" not in class_balance.columns:
    class_balance.columns = [QUADRANT_COL, "count"]
class_balance = (
    df[QUADRANT_COL]
    .value_counts()
    .rename_axis(QUADRANT_COL)
    .reset_index(name="count")
    .reindex(columns=[QUADRANT_COL, "count"])
)

class_balance["percent"] = class_balance["count"] / class_balance["count"].sum() * 100
class_balance["short"] = class_balance[QUADRANT_COL].map(QUADRANT_SHORT)
save_table(class_balance, "class_balance_fingerprints.csv")

fig = px.bar(
    class_balance,
    x="short", y="count",
    color=QUADRANT_COL,
    color_discrete_map={k: QUADRANT_COLOR_MAP.get(k, "#888") for k in class_balance[QUADRANT_COL]},
    text=class_balance["percent"].map(lambda x: f"{x:.1f}%"),
    title="Distribuição dos quadrantes — features de fingerprint",
)
fig.add_hline(
    y=len(df) / 4, line_dash="dash", line_color="gray",
    annotation_text="Equilíbrio ideal (25%)", annotation_position="top right",
)
fig.update_layout(xaxis_title="Quadrante", yaxis_title="Contagem", showlegend=False)
save_fig(fig, "class_balance_fingerprints")

display(class_balance[["short", "count", "percent"]])

Tabela salva: C:\dev\python\TCC Ajustado\Dados\pycaret_quadrants_fingerprints_outputs\tables\class_balance_fingerprints.csv


,short,count,percent
0,Q1 Alegre/Energ.,3217,49.446665
1,Q3 Triste/Melanc.,1382,21.241931
2,Q2 Tenso/Raivoso,1019,15.662465
3,Q4 Calmo/Relaxado,888,13.648939


In [9]:
# Espaço Valence–Arousal com quadrantes
sample_va = safe_sample(df, max_rows=8000)
sample_va["short"] = sample_va[QUADRANT_COL].map(QUADRANT_SHORT)

fig = px.scatter(
    sample_va,
    x=VALENCE_COL, y=AROUSAL_COL,
    color="short",
    color_discrete_map={v: QUADRANT_COLOR_MAP.get(k, "#888") for k, v in QUADRANT_SHORT.items()},
    opacity=0.45,
    title="Espaço Valence–Arousal (amostra de blocos de fingerprint)",
)
fig.add_vline(x=VALENCE_THRESHOLD, line_dash="dash", line_color="black")
fig.add_hline(y=AROUSAL_THRESHOLD, line_dash="dash", line_color="black")
fig.update_layout(xaxis_title="Valence", yaxis_title="Arousal", legend_title="Quadrante")
save_fig(fig, "va_space_fingerprints")

## 6. Inventário e ranking ANOVA das features de fingerprint

In [10]:
def _fingerprint_original_name(feature):
    """Remove prefixos de origem dos arquivos para classificar a feature real."""
    raw = str(feature).strip().lower()
    raw = re.sub(r"^(fpbandraw__|fpband__|fprank__)", "", raw)
    raw = re.sub(r"^fp_", "", raw)
    return raw


def fingerprint_technique(feature):
    raw = str(feature).strip().lower()
    if raw.startswith("fpbandraw__") or raw.startswith("fpband__"):
        return "band_rank"
    if raw.startswith("fprank__"):
        return "rank"
    return "desconhecida"


def fingerprint_origin(feature):
    full = str(feature).strip().lower()
    raw = _fingerprint_original_name(feature)
    if full.startswith("fpbandraw__"):
        return "raw_peaks"
    if re.match(r"^rp_", raw, re.IGNORECASE) or "peak" in raw:
        return "raw_peaks"
    return "bloco"


def fingerprint_band(feature):
    raw = _fingerprint_original_name(feature)
    if raw.startswith("raw_very_high") or raw.startswith("very_high") or "_very_high_" in raw or raw.endswith("_very_high") or "_vh_" in raw:
        return "very_high"
    if raw.startswith("raw_high") or raw.startswith("high") or "_high_" in raw or raw.endswith("_high"):
        return "high"
    if raw.startswith("raw_mid") or raw.startswith("mid") or "_mid_" in raw or raw.endswith("_mid"):
        return "mid"
    if raw.startswith("raw_low") or raw.startswith("low") or "_low_" in raw or raw.endswith("_low"):
        return "low"
    return "global"


def fingerprint_metric(feature):
    raw = _fingerprint_original_name(feature)
    if "freq" in raw:             return "frequency"
    if "mag" in raw or "energy" in raw:
                                  return "magnitude"
    if "rank" in raw:             return "rank"
    if "midi" in raw or "pitch" in raw or "octave" in raw or "semitone" in raw:
                                  return "pitch_octave"
    if "count" in raw or raw.endswith("_n") or "peak" in raw:
                                  return "counts"
    if "std" in raw:              return "dispersion"
    if "mean" in raw or "score" in raw:
                                  return "central_tendency"
    return "other"


inventory_rows = []
for feat in usable_feature_cols:
    inventory_rows.append({
        "feature": feat,
        "tecnica": fingerprint_technique(feat),
        "origem": fingerprint_origin(feat),
        "banda": fingerprint_band(feat),
        "metrica": fingerprint_metric(feat),
        "missing_rate": float(missing_rate.get(feat, np.nan)),
    })
inventory = pd.DataFrame(inventory_rows)
save_table(inventory, "feature_inventory_fingerprints.csv")

print("Distribuicao por tecnica:")
display(inventory["tecnica"].value_counts().to_frame())
print("Distribuicao por origem:")
display(inventory["origem"].value_counts().to_frame())
print("Distribuicao por tecnica x origem:")
display(inventory.groupby(["tecnica", "origem"]).size().to_frame("n_features"))
print("Distribuicao por banda:")
display(inventory["banda"].value_counts().to_frame())
print("Distribuicao por metrica:")
display(inventory["metrica"].value_counts().to_frame())


Tabela salva: C:\dev\python\TCC Ajustado\Dados\pycaret_quadrants_fingerprints_outputs\tables\feature_inventory_fingerprints.csv
Distribuicao por tecnica:


,count
tecnica,
band_rank,548
rank,207


Distribuicao por origem:


,count
origem,
bloco,506
raw_peaks,249


Distribuicao por tecnica x origem:


n_features
tecnica   origem               
band_rank bloco             302
          raw_peaks         246
rank      bloco             204
          raw_peaks           3

Distribuicao por banda:


,count
banda,
global,257
mid,127
high,124
very_high,124
low,123


Distribuicao por metrica:


,count
metrica,
pitch_octave,307
magnitude,232
frequency,125
counts,44
rank,17
other,16
central_tendency,9
dispersion,5


In [11]:
from sklearn.feature_selection import f_classif

X_anova = df[usable_feature_cols].apply(pd.to_numeric, errors="coerce")
X_anova = X_anova.fillna(X_anova.median())
y_anova = df[QUADRANT_COL].astype(str)

f_scores, _ = f_classif(X_anova, y_anova)

ranking = inventory.copy()
ranking["anova_f"] = f_scores
ranking = ranking.sort_values("anova_f", ascending=False).reset_index(drop=True)
save_table(ranking, "feature_ranking_fingerprints.csv")

display(Markdown("### Top-20 features por poder discriminativo (ANOVA F-score)"))
display(ranking[["feature", "banda", "metrica", "anova_f"]].head(20))

Tabela salva: C:\dev\python\TCC Ajustado\Dados\pycaret_quadrants_fingerprints_outputs\tables\feature_ranking_fingerprints.csv


### Top-20 features por poder discriminativo (ANOVA F-score)

,feature,banda,metrica,anova_f
0,fpband__energy_db_high,high,magnitude,990.493879
1,fprank__fp_magnitude_mean,global,magnitude,820.255112
2,fprank__fp_magnitude_mean_norm_block,global,magnitude,742.455630
3,fpband__fp_very_high_top_9_magnitude_db,very_high,magnitude,713.461314
4,fpband__fp_high_top_10_magnitude_db,high,magnitude,711.954089
5,fpband__fp_very_high_top_7_magnitude_db,very_high,magnitude,711.851367
6,fpband__fp_very_high_top_8_magnitude_db,very_high,magnitude,710.756098
7,fpband__fp_very_high_top_10_magnitude_db,very_high,magnitude,709.239731
8,fpband__fp_very_high_mag_mean_db,very_high,magnitude,708.201929
9,fpband__fp_very_high_top_5_magnitude_db,very_high,magnitude,706.266428


In [12]:
# Ranking ANOVA por banda
band_anova = (
    ranking.groupby("banda")
    .agg(n_features=("feature", "count"), anova_f_mean=("anova_f", "mean"), anova_f_max=("anova_f", "max"))
    .reset_index()
    .sort_values("anova_f_mean", ascending=False)
)
save_table(band_anova, "band_anova_summary_fingerprints.csv")

fig = px.bar(
    band_anova.sort_values("anova_f_mean"),
    x="anova_f_mean", y="banda", orientation="h",
    title="Poder discriminativo médio (ANOVA F) por banda de fingerprint",
    text=band_anova.sort_values("anova_f_mean")["anova_f_mean"].map(lambda x: f"{x:.1f}"),
)
fig.update_layout(xaxis_title="ANOVA F médio", yaxis_title="Banda")
save_fig(fig, "band_anova_fingerprints")

Tabela salva: C:\dev\python\TCC Ajustado\Dados\pycaret_quadrants_fingerprints_outputs\tables\band_anova_summary_fingerprints.csv


In [13]:
# Top-20 features — gráfico de barras
top20 = ranking.head(20).copy()
top20["feature_short"] = top20["feature"].str.replace(r"^fp_", "", regex=True)

fig = px.bar(
    top20.sort_values("anova_f"),
    x="anova_f", y="feature_short", orientation="h",
    color="banda",
    title="Top-20 features de fingerprint por ANOVA F-score",
    text=top20.sort_values("anova_f")["anova_f"].map(lambda x: f"{x:.1f}"),
)
fig.update_layout(xaxis_title="ANOVA F-score", yaxis_title="Feature", legend_title="Banda")
save_fig(fig, "top20_anova_fingerprints")

## 7. Validação oficial — GroupKFold por `song_id`

Mesma configuração do baseline openSMILE:
- `GroupKFold(n_splits=5)` com grupos por `song_id`
- Pipeline: `SimpleImputer → [SelectKBest] → StandardScaler → estimador`
- Métrica principal de seleção: **macro-F1 médio**

In [14]:
def build_candidate_models():
    return {
        "Dummy_most_frequent":       DummyClassifier(strategy="most_frequent"),
        "Dummy_stratified":          DummyClassifier(strategy="stratified", random_state=RANDOM_STATE),
        "LogisticRegression_balanced": LogisticRegression(
            max_iter=3000, class_weight="balanced", random_state=RANDOM_STATE,
        ),
        "RidgeClassifier_balanced":  RidgeClassifier(class_weight="balanced", random_state=RANDOM_STATE),
        "RandomForest_balanced":     RandomForestClassifier(
            n_estimators=250, class_weight="balanced_subsample", random_state=RANDOM_STATE, n_jobs=N_JOBS_MODEL,
        ),
        "ExtraTrees_balanced":       ExtraTreesClassifier(
            n_estimators=250, class_weight="balanced", random_state=RANDOM_STATE, n_jobs=N_JOBS_MODEL,
        ),
        "KNeighbors":                KNeighborsClassifier(n_neighbors=15, weights="distance"),
        "DecisionTree_balanced":     DecisionTreeClassifier(class_weight="balanced", random_state=RANDOM_STATE),
        "GaussianNB":                GaussianNB(),
    }


def build_pipeline(estimator, selector_k=None, n_features=None):
    steps = [("imputer", SimpleImputer(strategy="median"))]
    if selector_k is not None:
        k_eff = min(int(selector_k), int(n_features))
        steps.append(("selector", SelectKBest(score_func=f_classif, k=k_eff)))
    steps.append(("scaler", StandardScaler()))
    steps.append(("model", estimator))
    return Pipeline(steps)


def classification_metrics(y_true, y_pred):
    return {
        "accuracy":           accuracy_score(y_true, y_pred),
        "balanced_accuracy":  balanced_accuracy_score(y_true, y_pred),
        "macro_f1":           f1_score(y_true, y_pred, average="macro",    zero_division=0),
        "weighted_f1":        f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "macro_precision":    precision_score(y_true, y_pred, average="macro",  zero_division=0),
        "macro_recall":       recall_score(y_true, y_pred,    average="macro",  zero_division=0),
    }


def evaluate_pipeline_groupkfold(pipeline, X, y, groups, labels, cv):
    fold_rows = []
    oof_pred  = pd.Series(index=y.index, dtype="object")

    for fold_idx, (train_idx, test_idx) in enumerate(cv.split(X, y, groups), start=1):
        model = clone(pipeline)
        X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
        y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]
        model.fit(X_tr, y_tr)
        pred = model.predict(X_te)
        oof_pred.iloc[test_idx] = pred

        row = {
            "fold":            fold_idx,
            "n_train":         len(train_idx),
            "n_test":          len(test_idx),
            "n_groups_train":  pd.Series(groups.iloc[train_idx]).nunique(),
            "n_groups_test":   pd.Series(groups.iloc[test_idx]).nunique(),
        }
        row.update(classification_metrics(y_te, pred))
        fold_rows.append(row)

    return pd.DataFrame(fold_rows), oof_pred


def summarize_fold_metrics(fold_df):
    metric_cols = ["accuracy", "balanced_accuracy", "macro_f1", "weighted_f1", "macro_precision", "macro_recall"]
    return {f"{m}_mean": fold_df[m].mean() for m in metric_cols} | \
           {f"{m}_std":  fold_df[m].std(ddof=1) for m in metric_cols}


print("Funções de modelagem carregadas.")

Funções de modelagem carregadas.


In [15]:
X      = df[usable_feature_cols].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)
y      = df[QUADRANT_COL].astype(str)
groups = df[SONG_ID_COL].astype(str)

labels_present = [q for q in QUADRANT_ORDER if q in set(y)]
n_groups       = groups.nunique()
cv_splits      = min(N_SPLITS, n_groups)

if cv_splits < 2:
    raise RuntimeError("São necessários pelo menos 2 grupos de song_id para GroupKFold.")

cv = GroupKFold(n_splits=cv_splits)

print(f"Amostras: {len(X):,}  |  grupos (músicas): {n_groups}  |  folds: {cv_splits}")
print(f"Features: {len(usable_feature_cols)}  |  Classes: {labels_present}")

Amostras: 6,506  |  grupos (músicas): 1802  |  folds: 5
Features: 755  |  Classes: ['Q1_Alegre_Energetico', 'Q2_Tenso_Raivoso', 'Q3_Triste_Melancolico', 'Q4_Calmo_Relaxado']


In [16]:
models = build_candidate_models()
selector_options = [None] + [k for k in SELECTOR_K_VALUES if k < len(usable_feature_cols)]

results_rows       = []
fold_metric_frames = []
errors             = []

for model_name, estimator in tqdm(models.items(), desc="Modelos"):
    for selector_k in selector_options:
        sel_name = "none" if selector_k is None else f"SelectKBest(k={selector_k})"
        pipeline = build_pipeline(estimator, selector_k=selector_k, n_features=len(usable_feature_cols))
        try:
            fold_df, _ = evaluate_pipeline_groupkfold(pipeline, X, y, groups, labels_present, cv)
            fold_df.insert(0, "selector", sel_name)
            fold_df.insert(0, "model",    model_name)
            fold_metric_frames.append(fold_df)

            row = {
                "model":                model_name,
                "selector":             sel_name,
                "n_samples":            len(df),
                "n_groups":             n_groups,
                "n_features_requested": len(usable_feature_cols),
                "n_features_effective": len(usable_feature_cols) if selector_k is None else min(selector_k, len(usable_feature_cols)),
                "cv":                   f"GroupKFold({cv_splits}) por song_id",
                "feature_set":          "fingerprints",
                "status":               "ok",
            }
            row.update(summarize_fold_metrics(fold_df))
            results_rows.append(row)
        except Exception as e:
            errors.append({"model": model_name, "selector": sel_name, "error": str(e)})
            results_rows.append({
                "model": model_name, "selector": sel_name,
                "feature_set": "fingerprints", "status": f"error: {e}",
            })

clf_results = pd.DataFrame(results_rows).sort_values(
    ["status", "macro_f1_mean"], ascending=[False, False], na_position="last"
)
fold_metrics = pd.concat(fold_metric_frames, ignore_index=True) if fold_metric_frames else pd.DataFrame()

save_table(clf_results,  "classification_groupkfold_results_fingerprints.csv")
if not fold_metrics.empty:
    save_table(fold_metrics, "classification_groupkfold_fold_metrics_fingerprints.csv")

if errors:
    print(f"[AVISO] {len(errors)} erros durante a avaliação.")
    for e in errors:
        print(" ", e)

ok_results = clf_results[clf_results["status"].eq("ok")].copy()
if ok_results.empty:
    raise RuntimeError("Nenhum modelo foi avaliado com sucesso.")

best_row = ok_results.sort_values("macro_f1_mean", ascending=False).iloc[0]
print("\nMelhor modelo (fingerprints):")
display(best_row.to_frame().T)

Modelos:  67%|██████▋   | 6/9 [04:10<02:29, 49.88s/it]  File "c:\Users\felip\miniconda3\envs\pycaret311\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "c:\Users\felip\miniconda3\envs\pycaret311\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\felip\miniconda3\envs\pycaret311\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "c:\Users\felip\miniconda3\envs\pycaret311\Lib\subprocess.py", line 1538, in _execute_child
    hp, ht, pid, tid = _winapi.CreateProcess(executable, args,
                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
Modelos: 100%|██████████| 9/9 [05:45<00:00, 38.37s/it]

Tabela salva: C:\dev\python\TCC Ajustado\Dados\pycaret_quadrants_fingerprints_outputs\tables\classification_groupkfold_results_fingerprints.csv
Tabela salva: C:\dev\python\TCC Ajustado\Dados\pycaret_quadrants_fingerprints_outputs\tables\classification_groupkfold_fold_metrics_fingerprints.csv

Melhor modelo (fingerprints):


,model,selector,n_samples,n_groups,n_features_requested,n_features_effective,cv,feature_set,status,accuracy_mean,...,macro_f1_mean,weighted_f1_mean,macro_precision_mean,macro_recall_mean,accuracy_std,balanced_accuracy_std,macro_f1_std,weighted_f1_std,macro_precision_std,macro_recall_std
14,LogisticRegression_balanced,SelectKBest(k=200),6506,1802,755,200,GroupKFold(5) por song_id,fingerprints,ok,0.519836,...,0.468302,0.538993,0.469775,0.486396,0.016831,0.020366,0.020579,0.015175,0.020475,0.020366


In [17]:
# Gráfico: top modelos por macro-F1
non_dummy = ok_results[~ok_results["model"].str.startswith("Dummy")].copy()
dummy_rows = ok_results[ok_results["model"].str.startswith("Dummy")].copy()

fig = px.bar(
    non_dummy.head(25).sort_values("macro_f1_mean"),
    x="macro_f1_mean", y="model", color="selector", orientation="h",
    error_x="macro_f1_std",
    title="Top modelos — Fingerprints — GroupKFold por song_id",
    text=non_dummy.head(25).sort_values("macro_f1_mean")["macro_f1_mean"].map(lambda x: f"{x:.3f}"),
)
fig.update_layout(xaxis_title="Macro-F1 médio", yaxis_title="Modelo", legend_title="Seletor")
save_fig(fig, "classification_macro_f1_fingerprints")

## 8. Análise detalhada do melhor modelo

In [18]:
BEST_MODEL_NAME  = best_row["model"]
BEST_SELECTOR    = best_row["selector"]
BEST_SELECTOR_K  = None if BEST_SELECTOR == "none" else int(re.search(r"k=(\d+)", BEST_SELECTOR).group(1))

best_pipeline = build_pipeline(
    build_candidate_models()[BEST_MODEL_NAME],
    selector_k=BEST_SELECTOR_K,
    n_features=len(usable_feature_cols),
)
best_fold_df, best_oof_pred = evaluate_pipeline_groupkfold(
    best_pipeline, X, y, groups, labels_present, cv
)

# Relatório de classificação
report = classification_report(
    y, best_oof_pred.astype(str),
    labels=labels_present, output_dict=True, zero_division=0,
)
report_df = pd.DataFrame(report).T.reset_index().rename(columns={"index": "label"})
save_table(report_df, "classification_report_best_model_fingerprints.csv")

display(Markdown(f"### Relatório de classificação OOF — {BEST_MODEL_NAME} / {BEST_SELECTOR}"))
display(report_df)

Tabela salva: C:\dev\python\TCC Ajustado\Dados\pycaret_quadrants_fingerprints_outputs\tables\classification_report_best_model_fingerprints.csv


### Relatório de classificação OOF — LogisticRegression_balanced / SelectKBest(k=200)

,label,precision,recall,f1-score,support
0,Q1_Alegre_Energetico,0.770231,0.571029,0.655837,3217.000000
1,Q2_Tenso_Raivoso,0.358590,0.469087,0.406463,1019.000000
2,Q3_Triste_Melancolico,0.498275,0.522431,0.510067,1382.000000
3,Q4_Calmo_Relaxado,0.257655,0.388514,0.309834,888.000000
4,accuracy,0.519828,0.519828,0.519828,0.519828
5,macro avg,0.471187,0.487765,0.470550,6506.000000
6,weighted avg,0.578028,0.519828,0.538589,6506.000000


In [19]:
# Matriz de confusão
cm = confusion_matrix(y, best_oof_pred.astype(str), labels=labels_present)
cm_df = pd.DataFrame(cm, index=labels_present, columns=labels_present)
cm_norm = confusion_matrix(y, best_oof_pred.astype(str), labels=labels_present, normalize="true")
cm_norm_df = pd.DataFrame(cm_norm, index=labels_present, columns=labels_present)

save_table(cm_df.reset_index().rename(columns={"index": "actual"}), "confusion_matrix_best_model_fingerprints.csv")
save_table(cm_norm_df.reset_index().rename(columns={"index": "actual"}), "confusion_matrix_normalized_fingerprints.csv")

short_labels = [QUADRANT_SHORT.get(l, l) for l in labels_present]

fig = px.imshow(
    cm_norm_df.rename(index=QUADRANT_SHORT, columns=QUADRANT_SHORT),
    text_auto=".2f", aspect="auto", color_continuous_scale="Blues",
    title=f"Matriz de confusão normalizada OOF — {BEST_MODEL_NAME} (Fingerprints)",
)
fig.update_layout(xaxis_title="Predito", yaxis_title="Real")
save_fig(fig, "confusion_matrix_normalized_fingerprints")

Tabela salva: C:\dev\python\TCC Ajustado\Dados\pycaret_quadrants_fingerprints_outputs\tables\confusion_matrix_best_model_fingerprints.csv
Tabela salva: C:\dev\python\TCC Ajustado\Dados\pycaret_quadrants_fingerprints_outputs\tables\confusion_matrix_normalized_fingerprints.csv


In [20]:
# Análise fold a fold
display(Markdown("### Estabilidade por fold"))
display(best_fold_df[["fold", "n_train", "n_test", "accuracy", "balanced_accuracy", "macro_f1", "weighted_f1"]])

fig = go.Figure()
for metric, color in [("macro_f1", "#1565C0"), ("balanced_accuracy", "#2E7D32"), ("accuracy", "#C62828")]:
    fig.add_trace(go.Scatter(
        x=best_fold_df["fold"], y=best_fold_df[metric],
        mode="lines+markers", name=metric, line=dict(color=color),
    ))
fig.update_layout(
    title=f"Estabilidade por fold — {BEST_MODEL_NAME} (Fingerprints)",
    xaxis_title="Fold", yaxis_title="Métrica",
    legend_title="Métrica",
)
save_fig(fig, "fold_stability_fingerprints")

### Estabilidade por fold

,fold,n_train,n_test,accuracy,balanced_accuracy,macro_f1,weighted_f1
0,1,5203,1303,0.497314,0.453952,0.435082,0.520029
1,2,5205,1301,0.541891,0.507562,0.487036,0.559233
2,3,5205,1301,0.510377,0.482349,0.462542,0.528710
3,4,5205,1301,0.524212,0.490548,0.479221,0.545156
4,5,5206,1300,0.525385,0.497568,0.477631,0.541835


## 9. Comparação direta: Fingerprints vs openSMILE Baseline

Carrega os resultados do baseline openSMILE (`classification_groupkfold_results.csv`) e compara
os melhores resultados de cada representação na mesma tabela unificada.

In [21]:
baseline_path = BASELINE_TABLES_PATH / "classification_groupkfold_results.csv"

if baseline_path.exists():
    opensmile_results = pd.read_csv(baseline_path, encoding="utf-8-sig")
    opensmile_results["feature_set"] = "openSMILE"
    print(f"Baseline openSMILE carregado: {len(opensmile_results)} configurações")
else:
    print(f"[AVISO] Baseline não encontrado em {baseline_path}. Execute o notebook openSMILE primeiro.")
    opensmile_results = pd.DataFrame()

[AVISO] Baseline não encontrado em C:\dev\python\TCC Ajustado\Dados\pycaret_quadrants_outputs\tables\classification_groupkfold_results.csv. Execute o notebook openSMILE primeiro.


In [22]:
if not opensmile_results.empty:
    compare_cols = ["model", "selector", "feature_set",
                    "macro_f1_mean", "macro_f1_std",
                    "balanced_accuracy_mean", "accuracy_mean",
                    "n_features_effective", "cv", "status"]

    ok_os   = opensmile_results[opensmile_results["status"].eq("ok")].copy() if "status" in opensmile_results.columns else opensmile_results.copy()
    ok_fp   = ok_results.copy()

    combined = pd.concat(
        [ok_os[[c for c in compare_cols if c in ok_os.columns]],
         ok_fp[[c for c in compare_cols if c in ok_fp.columns]]],
        ignore_index=True,
    ).sort_values("macro_f1_mean", ascending=False)

    save_table(combined, "comparison_fingerprints_vs_opensmile.csv")

    # Melhor de cada representação
    non_dummy_combined = combined[~combined["model"].str.startswith("Dummy")]
    best_per_set = (
        non_dummy_combined
        .groupby("feature_set", sort=False)
        .apply(lambda g: g.sort_values("macro_f1_mean", ascending=False).iloc[0])
        .reset_index(drop=True)
    )
    save_table(best_per_set, "best_per_featureset_comparison.csv")

    display(Markdown("### Melhor resultado por representação (excluindo dummies)"))
    display(best_per_set[["feature_set", "model", "selector",
                           "macro_f1_mean", "macro_f1_std",
                           "balanced_accuracy_mean", "accuracy_mean",
                           "n_features_effective"]])

In [23]:
if not opensmile_results.empty:
    # Top-15 de cada representação para o gráfico lado a lado
    top_os = ok_os[~ok_os["model"].str.startswith("Dummy")].nlargest(15, "macro_f1_mean").copy()
    top_fp = ok_fp[~ok_fp["model"].str.startswith("Dummy")].nlargest(15, "macro_f1_mean").copy()
    top_combined = pd.concat([top_os, top_fp], ignore_index=True)
    top_combined["label"] = top_combined["model"] + " / " + top_combined["selector"]

    fig = px.scatter(
        top_combined,
        x="n_features_effective", y="macro_f1_mean",
        color="feature_set",
        symbol="feature_set",
        hover_data=["model", "selector", "macro_f1_std"],
        title="Macro-F1 vs. N° de features — Fingerprints vs openSMILE (Top-15 cada)",
    )
    fig.update_traces(marker_size=10)
    fig.update_layout(
        xaxis_title="Número de features efetivas",
        yaxis_title="Macro-F1 médio (GroupKFold)",
        legend_title="Representação",
    )
    save_fig(fig, "comparison_f1_vs_nfeatures")

    # Gráfico de barras agrupadas — melhor por representação
    fig2 = px.bar(
        best_per_set,
        x="feature_set", y="macro_f1_mean",
        color="feature_set",
        error_y="macro_f1_std",
        text=best_per_set["macro_f1_mean"].map(lambda x: f"{x:.3f}"),
        title="Melhor Macro-F1 por representação — quadrantes emocionais",
    )
    fig2.update_layout(
        xaxis_title="Representação", yaxis_title="Macro-F1 médio",
        showlegend=False,
    )
    save_fig(fig2, "comparison_best_macro_f1_barplot")

In [24]:
if not opensmile_results.empty:
    # Comparação normalizada: fingerprint / openSMILE para cada modelo comum
    os_best_per_model = (
        ok_os[~ok_os["model"].str.startswith("Dummy")]
        .sort_values("macro_f1_mean", ascending=False)
        .drop_duplicates(subset=["model"])
        .set_index("model")["macro_f1_mean"]
    )
    fp_best_per_model = (
        ok_fp[~ok_fp["model"].str.startswith("Dummy")]
        .sort_values("macro_f1_mean", ascending=False)
        .drop_duplicates(subset=["model"])
        .set_index("model")["macro_f1_mean"]
    )
    common_models = os_best_per_model.index.intersection(fp_best_per_model.index)

    if len(common_models) > 0:
        diff_df = pd.DataFrame({
            "model":        common_models,
            "openSMILE":    os_best_per_model[common_models].values,
            "fingerprints": fp_best_per_model[common_models].values,
        })
        diff_df["delta"] = diff_df["fingerprints"] - diff_df["openSMILE"]
        diff_df = diff_df.sort_values("delta", ascending=False)
        save_table(diff_df, "delta_macro_f1_per_model.csv")

        fig3 = px.bar(
            diff_df,
            x="delta", y="model", orientation="h",
            color=diff_df["delta"].map(lambda x: "Fingerprint melhor" if x >= 0 else "openSMILE melhor"),
            color_discrete_map={"Fingerprint melhor": "#2E7D32", "openSMILE melhor": "#C62828"},
            title="Delta Macro-F1: Fingerprints − openSMILE (mesmo modelo)",
            text=diff_df["delta"].map(lambda x: f"{x:+.3f}"),
        )
        fig3.add_vline(x=0, line_dash="dash", line_color="black")
        fig3.update_layout(
            xaxis_title="Δ Macro-F1 (Fingerprint − openSMILE)",
            yaxis_title="Modelo",
            showlegend=True,
            legend_title="",
        )
        save_fig(fig3, "delta_macro_f1_fingerprints_vs_opensmile")

        display(Markdown("### Delta Macro-F1 por modelo (Fingerprints − openSMILE)"))
        display(diff_df)

## 10. Comparação GroupKFold vs StratifiedGroupKFold (diagnóstico)

In [25]:
if StratifiedGroupKFold is not None:
    try:
        sgkf = StratifiedGroupKFold(n_splits=cv_splits)
        sgkf_fold_df, _ = evaluate_pipeline_groupkfold(
            clone(best_pipeline), X, y, groups, labels_present, sgkf
        )
        sgkf_summary = summarize_fold_metrics(sgkf_fold_df)

        cv_comp = pd.DataFrame([
            {
                "Validação":       f"GroupKFold({cv_splits})",
                "Macro-F1 médio":  round(float(best_row["macro_f1_mean"]), 4),
                "Macro-F1 std":    round(float(best_row["macro_f1_std"]),  4),
            },
            {
                "Validação":       f"StratifiedGroupKFold({cv_splits})",
                "Macro-F1 médio":  round(sgkf_summary["macro_f1_mean"], 4),
                "Macro-F1 std":    round(sgkf_summary["macro_f1_std"],  4),
            },
        ])
        save_table(cv_comp, "groupkfold_vs_stratified_fingerprints.csv")
        display(Markdown("### GroupKFold vs StratifiedGroupKFold (melhor modelo fingerprints)"))
        display(cv_comp)
    except Exception as e:
        print(f"[AVISO] StratifiedGroupKFold falhou: {e}")
else:
    print("StratifiedGroupKFold não disponível nesta versão do scikit-learn.")

Tabela salva: C:\dev\python\TCC Ajustado\Dados\pycaret_quadrants_fingerprints_outputs\tables\groupkfold_vs_stratified_fingerprints.csv


### GroupKFold vs StratifiedGroupKFold (melhor modelo fingerprints)

,Validação,Macro-F1 médio,Macro-F1 std
0,GroupKFold(5),0.4683,0.0206
1,StratifiedGroupKFold(5),0.4667,0.0268


## 6.5 Comparação com split holdout 70/20/10 por song_id

Protocolo alternativo ao `GroupKFold`: divisão única das músicas em **treino (70%) / teste (20%) / validação (10%)**, agrupadas por `song_id` para manter a restrição anti-vazamento.

A divisão é feita sobre os `song_id` únicos — nenhuma música aparece em mais de um split. Os resultados são comparados com o `GroupKFold(5)` da seção 7 para verificar consistência.

> **Critério de consistência:** |ΔMacro-F1| < 0.05 indica que o modelo é robusto ao protocolo de avaliação. Uma diferença maior pode indicar instabilidade ou viés na divisão única.

In [26]:
# =========================
# SPLIT HOLDOUT 70 / 20 / 10 POR song_id
# Divisão por músicas — mantém restrição anti-vazamento
# Protocolo alternativo ao GroupKFold(5) conforme sugerido pelo orientador
# =========================

if "X" not in dir() or "df" not in dir():
    print("[AVISO] Células da seção 6 não executadas — pulando holdout.")
else:
    from sklearn.base import clone as _sk_clone

    # ── 1. Dividir músicas ────────────────────────────────────────────────
    _all_songs = np.array(sorted(df[SONG_ID_COL].unique()))
    _rng = np.random.default_rng(RANDOM_STATE)
    _rng.shuffle(_all_songs)

    _n_total  = len(_all_songs)
    _n_train  = int(0.70 * _n_total)
    _n_test   = int(0.20 * _n_total)
    # validação = restante (~10%)

    _songs_train = set(_all_songs[:_n_train])
    _songs_test  = set(_all_songs[_n_train : _n_train + _n_test])
    _songs_val   = set(_all_songs[_n_train + _n_test :])

    _mask_tr = df[SONG_ID_COL].isin(_songs_train)
    _mask_te = df[SONG_ID_COL].isin(_songs_test)
    _mask_va = df[SONG_ID_COL].isin(_songs_val)

    _Xtr = df.loc[_mask_tr, usable_feature_cols].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)
    _Xte = df.loc[_mask_te, usable_feature_cols].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)
    _Xva = df.loc[_mask_va, usable_feature_cols].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)
    _ytr = df.loc[_mask_tr, QUADRANT_COL].astype(str)
    _yte = df.loc[_mask_te, QUADRANT_COL].astype(str)
    _yva = df.loc[_mask_va, QUADRANT_COL].astype(str)

    print(f"Split holdout por song_id  (random_state={RANDOM_STATE})")
    print(f"  Músicas — treino: {len(_songs_train):>4} ({len(_songs_train)/_n_total:.1%})  "
          f"teste: {len(_songs_test):>4} ({len(_songs_test)/_n_total:.1%})  "
          f"val: {len(_songs_val):>4} ({len(_songs_val)/_n_total:.1%})")
    print(f"  Blocos  — treino: {_Xtr.shape[0]:>6}  teste: {_Xte.shape[0]:>6}  val: {_Xva.shape[0]:>6}")
    print()

    # ── 2. Treinar todos os candidatos ────────────────────────────────────
    _holdout_rows = []
    _sel_opts = [None] + [k for k in SELECTOR_K_VALUES if k < len(usable_feature_cols)]

    for _mname, _estimator in tqdm(build_candidate_models().items(), desc="Holdout"):
        for _k in _sel_opts:
            try:
                _pipe = build_pipeline(_sk_clone(_estimator), selector_k=_k, n_features=len(usable_feature_cols))
                _pipe.fit(_Xtr, _ytr)
                for _split, _Xe, _ye, _ns in [
                    ("test",       _Xte, _yte, len(_songs_test)),
                    ("validation", _Xva, _yva, len(_songs_val)),
                ]:
                    _ypred = _pipe.predict(_Xe)
                    _m = classification_metrics(_ye, _ypred)
                    _holdout_rows.append({
                        "model":    _mname,
                        "selector": f"SelectKBest(k={_k})" if _k else "none",
                        "split":    _split,
                        "n_songs":  _ns,
                        "n_blocks": len(_ye),
                        "cv":       "holdout_70_20_10",
                        "status":   "ok",
                        **_m,
                    })
            except Exception as _exc:
                _holdout_rows.append({
                    "model": _mname,
                    "selector": f"SelectKBest(k={_k})" if _k else "none",
                    "split": "test", "status": f"error: {_exc}",
                })

    holdout_df = pd.DataFrame(_holdout_rows)
    save_table(holdout_df, "classification_holdout_70_20_10_results.csv")

    # ── 3. Melhor modelo no conjunto de teste ─────────────────────────────
    _test_ok = holdout_df[(holdout_df["split"] == "test") & (holdout_df["status"] == "ok")]
    _best_h  = _test_ok.sort_values("macro_f1", ascending=False).iloc[0]

    display(Markdown("### Melhor modelo — Holdout teste (20% das músicas)"))
    display(pd.DataFrame([{
        "Modelo":    _best_h["model"],
        "Seletor":   _best_h["selector"],
        "Macro-F1":  round(_best_h["macro_f1"], 4),
        "Bal. Acc.": round(_best_h["balanced_accuracy"], 4),
        "N. blocos": int(_best_h["n_blocks"]),
        "N. músicas": int(_best_h["n_songs"]),
    }]))

    # ── 4. Comparação GroupKFold vs Holdout ───────────────────────────────
    if "best_row" in dir():
        _kf_f1 = float(best_row.get("macro_f1_mean", float("nan")))
        _ho_f1 = float(_best_h["macro_f1"])
        _delta  = _ho_f1 - _kf_f1

        _cmp = pd.DataFrame([
            {
                "Protocolo": f"GroupKFold(5) — {best_row.get('model', '?')} / {best_row.get('selector', '?')}",
                "Macro-F1":  round(_kf_f1, 4),
                "Bal. Acc.": round(float(best_row.get("balanced_accuracy_mean", float("nan"))), 4),
                "Nota": "média de 5 folds",
            },
            {
                "Protocolo": f"Holdout teste — {_best_h['model']} / {_best_h['selector']}",
                "Macro-F1":  round(_ho_f1, 4),
                "Bal. Acc.": round(float(_best_h["balanced_accuracy"]), 4),
                "Nota": f"20% das músicas ({_best_h['n_songs']} músicas)",
            },
        ])
        save_table(_cmp, "classification_kfold_vs_holdout_comparison.csv")
        display(Markdown("### Comparação: GroupKFold(5) vs Holdout 70/20/10"))
        display(_cmp)
        print(f"  Δ Macro-F1 (holdout − kfold) = {_delta:+.4f}")

    # ── 5. Relatório por classe e matriz de confusão — conjunto de teste ──
    _best_k_val = None if _best_h["selector"] == "none" \
                  else int(re.search(r"k=(\d+)", _best_h["selector"]).group(1))
    _best_pipe_h = build_pipeline(
        build_candidate_models()[_best_h["model"]],
        selector_k=_best_k_val,
        n_features=len(usable_feature_cols),
    )
    _best_pipe_h.fit(_Xtr, _ytr)
    _ypred_te = _best_pipe_h.predict(_Xte)

    from sklearn.metrics import classification_report as _clf_report, confusion_matrix as _cm_fn

    _report_h = _clf_report(_yte, _ypred_te, labels=labels_present, output_dict=True, zero_division=0)
    _report_h_df = pd.DataFrame(_report_h).T.reset_index().rename(columns={"index": "label"})
    save_table(_report_h_df, "classification_report_holdout_test.csv")
    display(Markdown("### Relatório por quadrante — Holdout teste"))
    display(_report_h_df.round(3))

    _cm_norm = pd.DataFrame(
        _cm_fn(_yte, _ypred_te, labels=labels_present, normalize="true"),
        index=labels_present, columns=labels_present,
    ).round(3)
    save_table(_cm_norm.reset_index().rename(columns={"index": "actual"}),
               "confusion_matrix_holdout_test_normalized.csv")

    _fig_cm_h = px.imshow(
        _cm_norm, text_auto=True, aspect="auto", color_continuous_scale="Blues",
        title=f"Matriz de confusão normalizada — Holdout teste | {_best_h['model']} / {_best_h['selector']}",
    )
    _fig_cm_h.update_layout(xaxis_title="Predito", yaxis_title="Real")
    save_fig(_fig_cm_h, "confusion_matrix_holdout_test_normalized")
    print("[Holdout 70/20/10] concluído.")

Split holdout por song_id  (random_state=42)
  Músicas — treino: 1261 (70.0%)  teste:  360 (20.0%)  val:  181 (10.0%)
  Blocos  — treino:   4723  teste:   1183  val:    600



Holdout: 100%|██████████| 9/9 [01:02<00:00,  6.90s/it]

Tabela salva: C:\dev\python\TCC Ajustado\Dados\pycaret_quadrants_fingerprints_outputs\tables\classification_holdout_70_20_10_results.csv


### Melhor modelo — Holdout teste (20% das músicas)

,Modelo,Seletor,Macro-F1,Bal. Acc.,N. blocos,N. músicas
0,LogisticRegression_balanced,SelectKBest(k=100),0.4428,0.4829,1183,360


...

  Δ Macro-F1 (holdout − kfold) = -0.0255
Tabela salva: C:\dev\python\TCC Ajustado\Dados\pycaret_quadrants_fingerprints_outputs\tables\classification_report_holdout_test.csv


### Relatório por quadrante — Holdout teste

,label,precision,recall,f1-score,support
0,Q1_Alegre_Energetico,0.836,0.551,0.664,691.000
1,Q2_Tenso_Raivoso,0.191,0.333,0.243,123.000
2,Q3_Triste_Melancolico,0.502,0.532,0.517,235.000
3,Q4_Calmo_Relaxado,0.262,0.515,0.348,134.000
4,accuracy,0.521,0.521,0.521,0.521
5,macro avg,0.448,0.483,0.443,1183.000
6,weighted avg,0.637,0.521,0.555,1183.000


Tabela salva: C:\dev\python\TCC Ajustado\Dados\pycaret_quadrants_fingerprints_outputs\tables\confusion_matrix_holdout_test_normalized.csv


[Holdout 70/20/10] concluído.


## 11. Resumo final para o TCC

In [27]:
dummy_best = ok_results[ok_results["model"] == "Dummy_most_frequent"]["macro_f1_mean"].max() if "Dummy_most_frequent" in ok_results["model"].values else np.nan

summary_rows = [{
    "Experimento":           "Fingerprints (melhor)",
    "Modelo":                BEST_MODEL_NAME,
    "Seletor":               BEST_SELECTOR,
    "Macro-F1 médio":        round(float(best_row["macro_f1_mean"]), 4),
    "Macro-F1 std":          round(float(best_row["macro_f1_std"]),  4),
    "Balanced Acc médio":    round(float(best_row["balanced_accuracy_mean"]), 4),
    "N° features efetivas":  int(best_row["n_features_effective"]),
    "Protocolo CV":          f"GroupKFold({cv_splits}) por song_id",
}]

if not opensmile_results.empty and "best_per_set" in dir():
    os_best = best_per_set[best_per_set["feature_set"] == "openSMILE"]
    if not os_best.empty:
        r = os_best.iloc[0]
        summary_rows.append({
            "Experimento":           "openSMILE baseline",
            "Modelo":                r.get("model", ""),
            "Seletor":               r.get("selector", ""),
            "Macro-F1 médio":        round(float(r.get("macro_f1_mean", np.nan)), 4),
            "Macro-F1 std":          round(float(r.get("macro_f1_std",  np.nan)), 4),
            "Balanced Acc médio":    round(float(r.get("balanced_accuracy_mean", np.nan)), 4),
            "N° features efetivas":  int(r.get("n_features_effective", 0)),
            "Protocolo CV":          str(r.get("cv", "")),
        })

if not np.isnan(dummy_best):
    summary_rows.append({
        "Experimento":           "Dummy (most_frequent)",
        "Modelo":                "DummyClassifier",
        "Seletor":               "none",
        "Macro-F1 médio":        round(float(dummy_best), 4),
        "Macro-F1 std":          "",
        "Balanced Acc médio":    "",
        "N° features efetivas":  0,
        "Protocolo CV":          f"GroupKFold({cv_splits}) por song_id",
    })

summary_df = pd.DataFrame(summary_rows)
save_table(summary_df, "tcc_summary_fingerprints_vs_opensmile.csv")

display(Markdown("## Resumo para o TCC"))
display(summary_df)

Tabela salva: C:\dev\python\TCC Ajustado\Dados\pycaret_quadrants_fingerprints_outputs\tables\tcc_summary_fingerprints_vs_opensmile.csv


## Resumo para o TCC

,Experimento,Modelo,Seletor,Macro-F1 médio,Macro-F1 std,Balanced Acc médio,N° features efetivas,Protocolo CV
0,Fingerprints (melhor),LogisticRegression_balanced,SelectKBest(k=200),0.4683,0.0206,0.4864,200,GroupKFold(5) por song_id
1,Dummy (most_frequent),DummyClassifier,none,0.1653,,,0,GroupKFold(5) por song_id


In [28]:
# Treina modelo final em todos os dados e persiste
import joblib

final_model = clone(best_pipeline).fit(X, y)
model_artifact = {
    "model":              final_model,
    "features":           usable_feature_cols,
    "best_model_name":    BEST_MODEL_NAME,
    "best_selector":      BEST_SELECTOR,
    "quadrant_order":     QUADRANT_ORDER,
    "valence_threshold":  VALENCE_THRESHOLD,
    "arousal_threshold":  AROUSAL_THRESHOLD,
    "va_scale":           VA_SCALE,
    "feature_set":        "fingerprints",
}
model_path = MODELS_PATH / "best_fingerprint_quadrant_model.joblib"
joblib.dump(model_artifact, model_path)
print("Modelo salvo:", model_path)

Modelo salvo: C:\dev\python\TCC Ajustado\Dados\pycaret_quadrants_fingerprints_outputs\models\best_fingerprint_quadrant_model.joblib


## 12. Predições OOF e análise de erros por quadrante

In [29]:
oof_df = df[[SONG_ID_COL, BLOCK_ID_COL, START_COL, END_COL, VALENCE_COL, AROUSAL_COL, QUADRANT_COL]].copy()
oof_df["prediction"] = best_oof_pred.values
oof_df["correct"]    = oof_df[QUADRANT_COL].eq(oof_df["prediction"])
save_table(oof_df, "oof_predictions_best_model_fingerprints.csv")

# Taxa de acerto por quadrante real
acc_per_class = (
    oof_df.groupby(QUADRANT_COL)["correct"]
    .agg(["mean", "sum", "count"])
    .rename(columns={"mean": "accuracy", "sum": "n_correct", "count": "n_total"})
    .reset_index()
    .sort_values("accuracy", ascending=False)
)
acc_per_class["short"] = acc_per_class[QUADRANT_COL].map(QUADRANT_SHORT)
save_table(acc_per_class, "accuracy_per_class_fingerprints.csv")

fig = px.bar(
    acc_per_class,
    x="short", y="accuracy",
    color=QUADRANT_COL,
    color_discrete_map={k: QUADRANT_COLOR_MAP.get(k, "#888") for k in acc_per_class[QUADRANT_COL]},
    text=acc_per_class["accuracy"].map(lambda x: f"{x:.2%}"),
    title="Taxa de acerto por quadrante — melhor modelo (Fingerprints)",
)
fig.update_layout(
    xaxis_title="Quadrante", yaxis_title="Accuracy",
    yaxis_tickformat=".0%", showlegend=False,
)
save_fig(fig, "accuracy_per_class_fingerprints")

display(acc_per_class[["short", "accuracy", "n_correct", "n_total"]])

Tabela salva: C:\dev\python\TCC Ajustado\Dados\pycaret_quadrants_fingerprints_outputs\tables\oof_predictions_best_model_fingerprints.csv
Tabela salva: C:\dev\python\TCC Ajustado\Dados\pycaret_quadrants_fingerprints_outputs\tables\accuracy_per_class_fingerprints.csv


,short,accuracy,n_correct,n_total
0,Q1 Alegre/Energ.,0.571029,1837,3217
2,Q3 Triste/Melanc.,0.522431,722,1382
1,Q2 Tenso/Raivoso,0.469087,478,1019
3,Q4 Calmo/Relaxado,0.388514,345,888



## 13. openSMILE + Fingerprints — Feature Union (Teste Principal)

Combina as duas representações em um único vetor de features e avalia com o mesmo protocolo
`GroupKFold` por `song_id`. **Este é o experimento mais importante** do notebook: verifica se
a fusão supera cada representação isolada.

**Procedimento:**
1. Carrega as features openSMILE brutas (parquet por bloco) — busca automática em caminhos padrão.
2. Merge `inner` por `(song_id, block_idx)` com o dataset de fingerprints.
3. Avalia com os mesmos modelos e protocolo `GroupKFold(5)`.
4. Comparação tripla: **Fingerprints · openSMILE · openSMILE+Fingerprints**.


In [30]:

# Localiza e carrega a matriz openSMILE por bloco para fusão com fingerprints.
# Esta versão tenta primeiro caminhos manuais e depois datasets salvos pelo notebook openSMILE,
# incluindo quadrant_modeling_dataset.csv/parquet.

os_raw, os_raw_feat_cols, OPENSMILE_FEATURES_PATH, opensmile_audit = load_opensmile_features_for_fusion()
opensmile_raw_available = OPENSMILE_FEATURES_PATH is not None and not os_raw.empty and len(os_raw_feat_cols) > 0

save_table(opensmile_audit, "opensmile_feature_source_audit.csv")

if opensmile_raw_available:
    os_raw.columns = [str(c).strip() for c in os_raw.columns]
    os_raw = normalize_key_columns(os_raw)

    before_dedup = len(os_raw)
    os_raw = os_raw.drop_duplicates(subset=[SONG_ID_COL, BLOCK_ID_COL]).copy()
    after_dedup = len(os_raw)

    print(f"openSMILE carregado: {os_raw.shape[0]:,} linhas, {len(os_raw_feat_cols)} features úteis")
    print(f"Caminho selecionado: {OPENSMILE_FEATURES_PATH}")
    print(f"Duplicatas removidas por song_id/block_idx: {before_dedup - after_dedup:,}")

    os_leakage = make_leakage_report(os_raw_feat_cols, name="openSMILE_raw_for_fusion")
    save_table(os_leakage, "feature_leakage_check_opensmile_for_fusion.csv")
    display(os_leakage)
    if os_leakage["status"].iloc[0] != "ok":
        raise RuntimeError("Há colunas suspeitas nas features openSMILE. Revise feature_leakage_check_opensmile_for_fusion.csv.")
else:
    os_raw_feat_cols = []
    print("[AVISO] Features openSMILE brutas não encontradas. A fusão openSMILE+Fingerprint será ignorada.")
    print("Veja opensmile_feature_source_audit.csv para saber quais caminhos foram testados.")
    display(opensmile_audit.head(30))


Tabela salva: C:\dev\python\TCC Ajustado\Dados\pycaret_quadrants_fingerprints_outputs\tables\opensmile_feature_source_audit.csv
[AVISO] Features openSMILE brutas não encontradas. A fusão openSMILE+Fingerprint será ignorada.
Veja opensmile_feature_source_audit.csv para saber quais caminhos foram testados.


,path,exists,rows,cols,feature_cols,has_song_id,has_block_id,error
0,C:\dev\python\TCC Ajustado\Dados\pycaret_finge...,True,NaN,NaN,NaN,False,False,No columns to parse from file
1,C:\dev\python\TCC Ajustado\Dados\pycaret_finge...,True,120.0,18.0,10.0,False,False,NaN
2,C:\dev\python\TCC Ajustado\Dados\pycaret_finge...,True,24.0,19.0,11.0,False,False,NaN
3,C:\dev\python\TCC Ajustado\Dados\pycaret_quadr...,True,4.0,16.0,4.0,False,False,NaN
4,C:\dev\python\TCC Ajustado\Dados\pycaret_quadr...,True,35.0,21.0,13.0,False,False,NaN
5,C:\dev\python\TCC Ajustado\Dados\pycaret_quadr...,True,1.0,11.0,0.0,False,False,NaN
6,C:\dev\python\TCC Ajustado\Dados\pycaret_quadr...,True,7.0,8.0,3.0,False,False,NaN
7,C:\dev\python\TCC Ajustado\Dados\pycaret_quadr...,True,2.0,8.0,2.0,False,False,NaN


In [31]:

comb_results = pd.DataFrame()
ok_comb = pd.DataFrame()
comb_folds = []
combined_df = pd.DataFrame()
combined_feat_cols = []
os_cols_prefixed = []

if opensmile_raw_available:
    os_for_merge = (
        os_raw[[SONG_ID_COL, BLOCK_ID_COL] + os_raw_feat_cols]
        .copy()
        .add_prefix("os_")
        .rename(columns={f"os_{SONG_ID_COL}": SONG_ID_COL,
                         f"os_{BLOCK_ID_COL}": BLOCK_ID_COL})
    )
    os_cols_prefixed = [f"os_{c}" for c in os_raw_feat_cols if f"os_{c}" in os_for_merge.columns]

    combined_df = (
        df[[SONG_ID_COL, BLOCK_ID_COL, QUADRANT_COL] + usable_feature_cols]
        .merge(os_for_merge, on=[SONG_ID_COL, BLOCK_ID_COL], how="inner")
    )

    if combined_df.empty:
        print("[AVISO] Merge openSMILE+Fingerprints resultou em 0 linhas.")
        print("Verifique se song_id e block_idx usam a mesma granularidade nos dois datasets.")
    else:
        combined_feat_cols = [
            c for c in combined_df.columns
            if c not in {SONG_ID_COL, BLOCK_ID_COL, QUADRANT_COL}
            and not is_leakage_like_column(c)
        ]
        combined_feat_cols = [
            c for c in combined_feat_cols
            if pd.to_numeric(combined_df[c], errors="coerce").nunique(dropna=True) > 1
            and combined_df[c].isna().mean() <= MAX_MISSING_RATE
        ]

        comb_audit = pd.DataFrame([{
            "n_fingerprint_rows": len(df),
            "n_opensmile_rows": len(os_raw),
            "n_merged_rows": len(combined_df),
            "merge_coverage_vs_fingerprint_pct": 100 * len(combined_df) / max(len(df), 1),
            "n_fingerprint_features": len(usable_feature_cols),
            "n_opensmile_features": len(os_cols_prefixed),
            "n_combined_features": len(combined_feat_cols),
            "n_songs_merged": combined_df[SONG_ID_COL].nunique(),
        }])
        save_table(comb_audit, "fusion_merge_audit_opensmile_fingerprints.csv")
        display(Markdown("### Auditoria do merge openSMILE + Fingerprints"))
        display(comb_audit)

        combined_leakage = make_leakage_report(combined_feat_cols, name="openSMILE+Fingerprints")
        save_table(combined_leakage, "feature_leakage_check_combined.csv")
        display(combined_leakage)
        if combined_leakage["status"].iloc[0] != "ok":
            raise RuntimeError("Há colunas suspeitas no conjunto combinado. Revise feature_leakage_check_combined.csv.")

        print(f"Dataset combinado: {combined_df.shape[0]:,} amostras × {len(combined_feat_cols)} features")
        print(f"  Fingerprints: {len(usable_feature_cols)}  |  openSMILE: {len(os_cols_prefixed)}")

        X_comb   = (combined_df[combined_feat_cols]
                    .apply(pd.to_numeric, errors="coerce")
                    .replace([np.inf, -np.inf], np.nan))
        y_comb   = combined_df[QUADRANT_COL].astype(str)
        grp_comb = combined_df[SONG_ID_COL].astype(str)
        labels_comb = [q for q in QUADRANT_ORDER if q in set(y_comb)]
        cv_comb, cv_comb_splits = align_cv_for_dataset(combined_df)

        comb_rows, comb_folds = [], []
        for model_name, estimator in tqdm(build_candidate_models().items(), desc="openSMILE+FP"):
            for sel_k in [None] + [k for k in SELECTOR_K_VALUES if k < len(combined_feat_cols)]:
                sel_name = "none" if sel_k is None else f"SelectKBest(k={sel_k})"
                pipe = build_pipeline(estimator, selector_k=sel_k, n_features=len(combined_feat_cols))
                try:
                    fold_df, _ = evaluate_pipeline_groupkfold(
                        pipe, X_comb, y_comb, grp_comb, labels_comb, cv_comb
                    )
                    fold_df.insert(0, "selector", sel_name)
                    fold_df.insert(0, "model",    model_name)
                    fold_df.insert(0, "feature_set", "openSMILE+Fingerprints")
                    comb_folds.append(fold_df)
                    row = {
                        "model": model_name, "selector": sel_name,
                        "feature_set": "openSMILE+Fingerprints",
                        "n_samples": len(combined_df),
                        "n_groups": grp_comb.nunique(),
                        "n_features_effective": (
                            len(combined_feat_cols) if sel_k is None
                            else min(sel_k, len(combined_feat_cols))
                        ),
                        "cv": f"GroupKFold({cv_comb_splits}) por song_id",
                        "status": "ok",
                    }
                    row.update(summarize_fold_metrics(fold_df))
                    comb_rows.append(row)
                except Exception as e:
                    comb_rows.append({
                        "model": model_name, "selector": sel_name,
                        "feature_set": "openSMILE+Fingerprints", "status": f"error: {e}",
                    })

        comb_results = pd.DataFrame(comb_rows).sort_values(
            ["status", "macro_f1_mean"], ascending=[False, False], na_position="last"
        )
        save_table(comb_results, "classification_groupkfold_results_combined.csv")
        if comb_folds:
            save_table(
                pd.concat(comb_folds, ignore_index=True),
                "classification_groupkfold_fold_metrics_combined.csv",
            )

        ok_comb = comb_results[comb_results["status"].eq("ok")]
        _non_dummy_comb = ok_comb[~ok_comb["model"].astype(str).str.startswith("Dummy")]
        if not _non_dummy_comb.empty:
            best_comb = _non_dummy_comb.sort_values("macro_f1_mean", ascending=False).iloc[0]
            print("\nMelhor modelo (openSMILE+Fingerprints):")
            display(best_comb.to_frame().T)
        else:
            print("[AVISO] Nenhum modelo combinado não-Dummy com status ok.")
else:
    print("[SKIP] openSMILE não disponível — seção 13 ignorada.")


[SKIP] openSMILE não disponível — seção 13 ignorada.


In [32]:

# Comparação tripla: Fingerprints × openSMILE × openSMILE+Fingerprints
_compare_parts = []

if not opensmile_results.empty:
    _os_ok2 = (opensmile_results if "status" not in opensmile_results.columns
               else opensmile_results[opensmile_results["status"].eq("ok")])
    _nd_os2 = _os_ok2[~_os_ok2["model"].str.startswith("Dummy")]
    if not _nd_os2.empty:
        _compare_parts.append(
            _nd_os2.nlargest(1, "macro_f1_mean").assign(feature_set="openSMILE")
        )

_nd_fp2 = ok_results[~ok_results["model"].str.startswith("Dummy")]
if not _nd_fp2.empty:
    _compare_parts.append(_nd_fp2.nlargest(1, "macro_f1_mean").assign(feature_set="Fingerprints"))

if not ok_comb.empty:
    _nd_comb2 = ok_comb[~ok_comb["model"].str.startswith("Dummy")]
    if not _nd_comb2.empty:
        _compare_parts.append(_nd_comb2.nlargest(1, "macro_f1_mean"))

if _compare_parts:
    tripla_df = pd.concat(_compare_parts, ignore_index=True)
    save_table(tripla_df, "best_per_featureset_tripla.csv")

    display(Markdown("### Melhor resultado por representação — comparação tripla"))
    _show_cols = [c for c in ["feature_set", "model", "selector", "macro_f1_mean",
                               "macro_f1_std", "balanced_accuracy_mean", "accuracy_mean",
                               "n_features_effective"] if c in tripla_df.columns]
    display(tripla_df[_show_cols])

    fig_tripla = px.bar(
        tripla_df,
        x="feature_set", y="macro_f1_mean",
        color="feature_set",
        error_y="macro_f1_std" if "macro_f1_std" in tripla_df.columns else None,
        text=tripla_df["macro_f1_mean"].map(lambda x: f"{x:.3f}"),
        title="Macro-F1 por representação — Fingerprints vs openSMILE vs Fusão",
    )
    fig_tripla.update_layout(
        xaxis_title="Representação", yaxis_title="Macro-F1 médio", showlegend=False
    )
    save_fig(fig_tripla, "comparison_tripla_macro_f1")
else:
    print("[INFO] openSMILE não disponível — comparação tripla parcial (só Fingerprints).")


Tabela salva: C:\dev\python\TCC Ajustado\Dados\pycaret_quadrants_fingerprints_outputs\tables\best_per_featureset_tripla.csv


### Melhor resultado por representação — comparação tripla

,feature_set,model,selector,macro_f1_mean,macro_f1_std,balanced_accuracy_mean,accuracy_mean,n_features_effective
0,Fingerprints,LogisticRegression_balanced,SelectKBest(k=200),0.468302,0.020579,0.486396,0.519836,200


## 13.1 openSMILE + Fingerprints — fusões controladas

Além da fusão completa, esta seção testa combinações menores e interpretáveis:
- openSMILE + top-k fingerprints por ANOVA;
- openSMILE + fingerprints por banda;
- openSMILE + fingerprints por tipo;
- openSMILE + fingerprints sem redundância.

Também testa `openSMILE_mesma_intersecao` para comparar as fusões no mesmo conjunto de blocos.


In [33]:

fusion_variant_results = pd.DataFrame()
fusion_variant_folds = pd.DataFrame()


def evaluate_feature_set_classification(feature_set_name, data_df, feature_cols, y_col=QUADRANT_COL, group_col=SONG_ID_COL):
    feature_cols = [c for c in list(dict.fromkeys(feature_cols)) if c in data_df.columns and not is_leakage_like_column(c)]
    if not feature_cols:
        return pd.DataFrame([{
            "feature_set": feature_set_name,
            "model": "",
            "selector": "",
            "status": "error: no_valid_features",
        }]), pd.DataFrame()

    local_df = data_df[[group_col, y_col] + feature_cols].dropna(subset=[group_col, y_col]).copy()
    X_local = local_df[feature_cols].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)
    y_local = local_df[y_col].astype(str)
    groups_local = local_df[group_col].astype(str)
    labels_local = [q for q in QUADRANT_ORDER if q in set(y_local)]
    cv_local, cv_local_splits = align_cv_for_dataset(local_df)

    rows, folds = [], []
    for model_name, estimator in build_candidate_models().items():
        selector_candidates = [None] + [k for k in SELECTOR_K_VALUES if k < len(feature_cols)]
        for selector_k in selector_candidates:
            sel_name = "none" if selector_k is None else f"SelectKBest(k={selector_k})"
            pipe = build_pipeline(estimator, selector_k=selector_k, n_features=len(feature_cols))
            try:
                fold_df, _ = evaluate_pipeline_groupkfold(pipe, X_local, y_local, groups_local, labels_local, cv_local)
                fold_df.insert(0, "selector", sel_name)
                fold_df.insert(0, "model", model_name)
                fold_df.insert(0, "feature_set", feature_set_name)
                folds.append(fold_df)
                row = {
                    "feature_set": feature_set_name,
                    "model": model_name,
                    "selector": sel_name,
                    "n_samples": len(local_df),
                    "n_groups": groups_local.nunique(),
                    "n_features_effective": len(feature_cols) if selector_k is None else min(selector_k, len(feature_cols)),
                    "cv": f"GroupKFold({cv_local_splits}) por song_id",
                    "status": "ok",
                }
                row.update(summarize_fold_metrics(fold_df))
                rows.append(row)
            except Exception as e:
                rows.append({
                    "feature_set": feature_set_name,
                    "model": model_name,
                    "selector": sel_name,
                    "status": f"error: {e}",
                })
    return pd.DataFrame(rows), (pd.concat(folds, ignore_index=True) if folds else pd.DataFrame())


if RUN_FUSION_VARIANTS and opensmile_raw_available and not combined_df.empty:
    variant_map = {}

    ranked_fp_cols = []
    if "ranking" in globals() and isinstance(ranking, pd.DataFrame) and not ranking.empty:
        ranked_fp_cols = [c for c in ranking.sort_values("anova_f", ascending=False)["feature"].tolist() if c in usable_feature_cols]
    if not ranked_fp_cols:
        ranked_fp_cols = list(usable_feature_cols)

    if "uncorr_feature_cols" in globals() and uncorr_feature_cols:
        variant_map["openSMILE+FP_sem_redundancia"] = os_cols_prefixed + [c for c in uncorr_feature_cols if c in combined_df.columns]

    for k in FUSION_TOP_K_VALUES:
        cols = ranked_fp_cols[:min(k, len(ranked_fp_cols))]
        if cols:
            variant_map[f"openSMILE+FP_top{k}"] = os_cols_prefixed + cols

    for band in FUSION_BANDS:
        cols = [c for c in usable_feature_cols if fingerprint_band(c) == band]
        if cols:
            variant_map[f"openSMILE+FP_banda_{band}"] = os_cols_prefixed + cols

    for metric in FUSION_METRICS:
        cols = [c for c in usable_feature_cols if fingerprint_metric(c) == metric]
        if cols:
            variant_map[f"openSMILE+FP_tipo_{metric}"] = os_cols_prefixed + cols

    # Baseline openSMILE na mesma interseção de blocos da fusão.
    variant_map["openSMILE_mesma_intersecao"] = os_cols_prefixed

    variant_rows, variant_folds = [], []
    variant_inventory_rows = []
    for variant_name, variant_cols in tqdm(variant_map.items(), desc="Fusões controladas"):
        variant_cols = [c for c in list(dict.fromkeys(variant_cols)) if c in combined_df.columns]
        variant_inventory_rows.append({
            "feature_set": variant_name,
            "n_features": len(variant_cols),
            "n_opensmile_features": len([c for c in variant_cols if c.startswith("os_")]),
            "n_fingerprint_features": len([c for c in variant_cols if not c.startswith("os_")]),
        })
        res_v, folds_v = evaluate_feature_set_classification(variant_name, combined_df, variant_cols)
        variant_rows.append(res_v)
        if not folds_v.empty:
            variant_folds.append(folds_v)

    fusion_variant_inventory = pd.DataFrame(variant_inventory_rows)
    save_table(fusion_variant_inventory, "fusion_variants_inventory.csv")

    fusion_variant_results = pd.concat(variant_rows, ignore_index=True) if variant_rows else pd.DataFrame()
    fusion_variant_results = fusion_variant_results.sort_values(["status", "macro_f1_mean"], ascending=[False, False], na_position="last")
    fusion_variant_folds = pd.concat(variant_folds, ignore_index=True) if variant_folds else pd.DataFrame()
    save_table(fusion_variant_results, "classification_groupkfold_results_fusion_variants.csv")
    if not fusion_variant_folds.empty:
        save_table(fusion_variant_folds, "classification_groupkfold_fold_metrics_fusion_variants.csv")

    best_variants = (
        fusion_variant_results[fusion_variant_results["status"].eq("ok")]
        .loc[lambda d: ~d["model"].astype(str).str.startswith("Dummy")]
        .sort_values(["macro_f1_mean", "balanced_accuracy_mean"], ascending=False)
        .groupby("feature_set", as_index=False)
        .first()
        .sort_values("macro_f1_mean", ascending=False)
    )
    save_table(best_variants, "fusion_variants_best_by_feature_set.csv")
    display(Markdown("### Melhores fusões controladas"))
    display(best_variants.head(30))
else:
    print("Fusões controladas ignoradas: RUN_FUSION_VARIANTS=False, openSMILE indisponível ou merge vazio.")


Fusões controladas ignoradas: RUN_FUSION_VARIANTS=False, openSMILE indisponível ou merge vazio.


## 13.2 Tabela final de decisão — fingerprints vs openSMILE vs fusões

Esta é a tabela principal para o TCC: ela responde se o fingerprint isolado ou combinado melhora o baseline openSMILE.


In [34]:

def _prepare_result_block(df_src, feature_set_name):
    if df_src is None or df_src.empty:
        return pd.DataFrame()
    block = df_src.copy()
    if "status" in block.columns:
        block = block[block["status"].eq("ok")].copy()
    if block.empty:
        return pd.DataFrame()
    if "model" in block.columns:
        block = block[~block["model"].astype(str).str.startswith("Dummy")].copy()
    if block.empty:
        return pd.DataFrame()

    block["feature_set"] = feature_set_name
    if "n_features_effective" not in block.columns:
        block["n_features_effective"] = block["n_features"] if "n_features" in block.columns else np.nan
    return block


comparison_parts = [
    _prepare_result_block(globals().get("ok_os", opensmile_results), "openSMILE"),
    _prepare_result_block(globals().get("ok_results", pd.DataFrame()), "Fingerprints"),
    _prepare_result_block(globals().get("corr_filter_results", pd.DataFrame()), "Fingerprints sem redundância"),
    _prepare_result_block(globals().get("ok_comb", pd.DataFrame()), "openSMILE+Fingerprints"),
    _prepare_result_block(globals().get("fusion_variant_results", pd.DataFrame()), "Fusões controladas"),
]
comparison_parts = [x for x in comparison_parts if x is not None and not x.empty]

if comparison_parts:
    final_compare_df = pd.concat(comparison_parts, ignore_index=True, sort=False)
    final_compare_df = final_compare_df.sort_values(
        [c for c in ["macro_f1_mean", "balanced_accuracy_mean", "accuracy_mean"] if c in final_compare_df.columns],
        ascending=[False] * len([c for c in ["macro_f1_mean", "balanced_accuracy_mean", "accuracy_mean"] if c in final_compare_df.columns]),
        na_position="last",
    ).reset_index(drop=True)

    save_table(final_compare_df, "comparison_complete_fingerprints_vs_opensmile.csv")

    best_open_smile_row = final_compare_df[final_compare_df["feature_set"].eq("openSMILE")].head(1)
    open_smile_best_macro_f1 = float(best_open_smile_row.iloc[0]["macro_f1_mean"]) if not best_open_smile_row.empty else np.nan

    summary_rows = []
    for feature_set_name in final_compare_df["feature_set"].dropna().unique().tolist():
        subset = final_compare_df[final_compare_df["feature_set"].eq(feature_set_name)].copy()
        if subset.empty:
            continue
        best_row_local = subset.sort_values(
            [c for c in ["macro_f1_mean", "balanced_accuracy_mean", "accuracy_mean"] if c in subset.columns],
            ascending=[False] * len([c for c in ["macro_f1_mean", "balanced_accuracy_mean", "accuracy_mean"] if c in subset.columns]),
            na_position="last",
        ).iloc[0]

        if feature_set_name == "openSMILE":
            delta = 0.0
        elif pd.notna(open_smile_best_macro_f1):
            delta = float(best_row_local.get("macro_f1_mean", np.nan)) - open_smile_best_macro_f1
        else:
            delta = np.nan

        summary_rows.append({
            "feature_set": feature_set_name,
            "model": best_row_local.get("model", ""),
            "selector": best_row_local.get("selector", ""),
            "macro_f1_mean": float(best_row_local.get("macro_f1_mean", np.nan)),
            "macro_f1_std": float(best_row_local.get("macro_f1_std", np.nan)),
            "balanced_accuracy_mean": float(best_row_local.get("balanced_accuracy_mean", np.nan)),
            "accuracy_mean": float(best_row_local.get("accuracy_mean", np.nan)),
            "n_features_effective": best_row_local.get("n_features_effective", np.nan),
            "delta_macro_f1_vs_openSMILE": delta,
            "melhora_openSMILE": "sim" if pd.notna(delta) and delta > 0 else ("nao" if pd.notna(delta) else "indisponivel"),
            "status": best_row_local.get("status", "ok"),
        })

    final_summary_df = pd.DataFrame(summary_rows)
    save_table(final_summary_df, "final_comparison_summary_fingerprints_vs_opensmile.csv")

    display(Markdown("### Tabela final completa"))
    show_cols = [c for c in [
        "feature_set", "model", "selector", "macro_f1_mean", "macro_f1_std",
        "balanced_accuracy_mean", "accuracy_mean", "n_features_effective", "cv", "status",
    ] if c in final_compare_df.columns]
    display(final_compare_df[show_cols].head(30))

    display(Markdown("### Resumo final por representação"))
    display(final_summary_df)

    fig_final = px.bar(
        final_summary_df.sort_values("macro_f1_mean"),
        x="macro_f1_mean", y="feature_set", orientation="h",
        color="melhora_openSMILE",
        text=final_summary_df.sort_values("macro_f1_mean")["macro_f1_mean"].map(lambda x: f"{x:.3f}"),
        title="Resumo final — Macro-F1 por representação",
    )
    fig_final.update_layout(xaxis_title="Macro-F1 médio", yaxis_title="Representação")
    save_fig(fig_final, "final_comparison_summary_macro_f1")
else:
    final_compare_df = pd.DataFrame()
    final_summary_df = pd.DataFrame()
    print("[AVISO] Nenhum resultado válido encontrado para montar a tabela final.")


Tabela salva: C:\dev\python\TCC Ajustado\Dados\pycaret_quadrants_fingerprints_outputs\tables\comparison_complete_fingerprints_vs_opensmile.csv
Tabela salva: C:\dev\python\TCC Ajustado\Dados\pycaret_quadrants_fingerprints_outputs\tables\final_comparison_summary_fingerprints_vs_opensmile.csv


### Tabela final completa

,feature_set,model,selector,macro_f1_mean,macro_f1_std,balanced_accuracy_mean,accuracy_mean,n_features_effective,cv,status
0,Fingerprints,LogisticRegression_balanced,SelectKBest(k=200),0.468302,0.020579,0.486396,0.519836,200,GroupKFold(5) por song_id,ok
1,Fingerprints,RidgeClassifier_balanced,SelectKBest(k=200),0.456543,0.023758,0.474379,0.509385,200,GroupKFold(5) por song_id,ok
2,Fingerprints,GaussianNB,SelectKBest(k=200),0.450541,0.020778,0.452488,0.531366,200,GroupKFold(5) por song_id,ok
3,Fingerprints,LogisticRegression_balanced,SelectKBest(k=100),0.445932,0.029117,0.467929,0.487249,100,GroupKFold(5) por song_id,ok
4,Fingerprints,LogisticRegression_balanced,none,0.443253,0.027993,0.458887,0.496015,755,GroupKFold(5) por song_id,ok
5,Fingerprints,RidgeClassifier_balanced,none,0.442608,0.022090,0.459326,0.495705,755,GroupKFold(5) por song_id,ok
6,Fingerprints,LogisticRegression_balanced,SelectKBest(k=60),0.439195,0.033238,0.461524,0.480021,60,GroupKFold(5) por song_id,ok
7,Fingerprints,RidgeClassifier_balanced,SelectKBest(k=100),0.437682,0.024742,0.458441,0.483561,100,GroupKFold(5) por song_id,ok
8,Fingerprints,GaussianNB,none,0.437232,0.012884,0.450431,0.494458,755,GroupKFold(5) por song_id,ok
9,Fingerprints,RandomForest_balanced,SelectKBest(k=200),0.433171,0.023985,0.437822,0.595913,200,GroupKFold(5) por song_id,ok


### Resumo final por representação

,feature_set,model,selector,macro_f1_mean,macro_f1_std,balanced_accuracy_mean,accuracy_mean,n_features_effective,delta_macro_f1_vs_openSMILE,melhora_openSMILE,status
0,Fingerprints,LogisticRegression_balanced,SelectKBest(k=200),0.468302,0.020579,0.486396,0.519836,200,NaN,indisponivel,ok


## 13.3 Teste pareado fold a fold

Quando os folds do openSMILE estão disponíveis, esta seção compara o ganho médio de Macro-F1 fold a fold.
Para fusões, a comparação mais justa é contra `openSMILE_mesma_intersecao`.


In [35]:

def _best_fold_subset(results_df, folds_df, label):
    if results_df is None or results_df.empty or folds_df is None or folds_df.empty:
        return pd.DataFrame()
    b = best_non_dummy(results_df)
    if b.empty:
        return pd.DataFrame()
    f = folds_df.copy()
    mask = f["model"].astype(str).eq(str(b.get("model"))) & f["selector"].astype(str).eq(str(b.get("selector")))
    if "feature_set" in f.columns and "feature_set" in b.index:
        mask = mask & f["feature_set"].astype(str).eq(str(b.get("feature_set")))
    out = f.loc[mask].copy()
    out["feature_set_comp"] = label
    return out


paired_rows = []
os_folds = pd.DataFrame()
os_fold_path = BASELINE_TABLES_PATH / "classification_groupkfold_fold_metrics.csv"
if os_fold_path.exists():
    os_folds = pd.read_csv(os_fold_path, encoding="utf-8-sig")

comb_folds_df = pd.concat(comb_folds, ignore_index=True) if "comb_folds" in globals() and isinstance(comb_folds, list) and comb_folds else pd.DataFrame()

fold_sources = {
    "Fingerprints": (ok_results, fold_metrics),
    "Fingerprints sem redundância": (globals().get("corr_filter_results", pd.DataFrame()), globals().get("corr_filter_folds", pd.DataFrame())),
    "openSMILE+Fingerprints": (globals().get("ok_comb", pd.DataFrame()), comb_folds_df),
    "Fusões controladas": (globals().get("fusion_variant_results", pd.DataFrame()), globals().get("fusion_variant_folds", pd.DataFrame())),
}

# Comparação contra baseline openSMILE oficial
os_best_fold = _best_fold_subset(opensmile_results, os_folds, "openSMILE") if not opensmile_results.empty else pd.DataFrame()
if not os_best_fold.empty:
    os_ref = os_best_fold[["fold", "macro_f1"]].rename(columns={"macro_f1": "macro_f1_openSMILE"})
    for label, (res_df, fold_df) in fold_sources.items():
        fbest = _best_fold_subset(res_df, fold_df, label)
        if fbest.empty:
            continue
        comp = fbest[["fold", "macro_f1"]].merge(os_ref, on="fold", how="inner")
        if comp.empty:
            continue
        diffs = comp["macro_f1"] - comp["macro_f1_openSMILE"]
        row = {
            "reference": "openSMILE_oficial",
            "comparison": f"{label} - openSMILE",
            "n_folds": len(comp),
            "delta_macro_f1_mean": diffs.mean(),
            "wins_vs_openSMILE": int((diffs > 0).sum()),
            "losses_vs_openSMILE": int((diffs < 0).sum()),
        }
        if len(comp) >= 3 and wilcoxon is not None:
            try:
                row["wilcoxon_p"] = wilcoxon(diffs).pvalue
            except Exception:
                row["wilcoxon_p"] = np.nan
        if len(comp) >= 3 and ttest_rel is not None:
            try:
                row["ttest_p"] = ttest_rel(comp["macro_f1"], comp["macro_f1_openSMILE"]).pvalue
            except Exception:
                row["ttest_p"] = np.nan
        paired_rows.append(row)

# Comparação de fusões contra openSMILE na mesma interseção
os_same_intersection_results = pd.DataFrame()
os_same_intersection_folds = pd.DataFrame()
if "fusion_variant_results" in globals() and not fusion_variant_results.empty:
    os_same_intersection_results = fusion_variant_results[
        fusion_variant_results["feature_set"].astype(str).eq("openSMILE_mesma_intersecao")
    ].copy()
if "fusion_variant_folds" in globals() and not fusion_variant_folds.empty:
    os_same_intersection_folds = fusion_variant_folds[
        fusion_variant_folds["feature_set"].astype(str).eq("openSMILE_mesma_intersecao")
    ].copy()

os_intersection_best_fold = _best_fold_subset(os_same_intersection_results, os_same_intersection_folds, "openSMILE_mesma_intersecao")
if not os_intersection_best_fold.empty:
    os_ref = os_intersection_best_fold[["fold", "macro_f1"]].rename(columns={"macro_f1": "macro_f1_openSMILE_intersecao"})
    for label, (res_df, fold_df) in {
        "openSMILE+Fingerprints": (globals().get("ok_comb", pd.DataFrame()), comb_folds_df),
        "Fusões controladas": (globals().get("fusion_variant_results", pd.DataFrame()), globals().get("fusion_variant_folds", pd.DataFrame())),
    }.items():
        fbest = _best_fold_subset(res_df, fold_df, label)
        if fbest.empty:
            continue
        comp = fbest[["fold", "macro_f1"]].merge(os_ref, on="fold", how="inner")
        if comp.empty:
            continue
        diffs = comp["macro_f1"] - comp["macro_f1_openSMILE_intersecao"]
        paired_rows.append({
            "reference": "openSMILE_mesma_intersecao",
            "comparison": f"{label} - openSMILE_mesma_intersecao",
            "n_folds": len(comp),
            "delta_macro_f1_mean": diffs.mean(),
            "wins_vs_openSMILE": int((diffs > 0).sum()),
            "losses_vs_openSMILE": int((diffs < 0).sum()),
            "wilcoxon_p": wilcoxon(diffs).pvalue if len(comp) >= 3 and wilcoxon is not None else np.nan,
            "ttest_p": ttest_rel(comp["macro_f1"], comp["macro_f1_openSMILE_intersecao"]).pvalue if len(comp) >= 3 and ttest_rel is not None else np.nan,
        })

paired_comparison_df = pd.DataFrame(paired_rows)
if not paired_comparison_df.empty:
    save_table(paired_comparison_df, "paired_fold_comparison_vs_opensmile.csv")
    display(Markdown("### Comparação pareada por fold contra openSMILE"))
    display(paired_comparison_df)
else:
    print("Comparação pareada indisponível: folds do openSMILE não encontrados ou incompatíveis.")


Comparação pareada indisponível: folds do openSMILE não encontrados ou incompatíveis.


In [36]:

# ============================================================
# Helper compartilhado — ablation study por subconjunto
# Reutilizado pelas seções 14, 15 e 16
# ============================================================

def run_ablation_study(feature_groups, X_full, y, groups, cv, study_name,
                       ablation_model_names=None):
    """
    Avalia cada subconjunto de features de forma isolada com GroupKFold.

    feature_groups       : dict {label: [col, ...]}
    ablation_model_names : lista de chaves de build_candidate_models()
                          (padrão: LR + RF + ExtraTrees)
    Retorna DataFrame com resultados por grupo × modelo.
    """
    if ablation_model_names is None:
        ablation_model_names = [
            "LogisticRegression_balanced",
            "RandomForest_balanced",
            "ExtraTrees_balanced",
        ]
    _candidate = build_candidate_models()
    _ablation_models = {k: _candidate[k] for k in ablation_model_names if k in _candidate}

    rows = []
    for group_label, feat_cols in feature_groups.items():
        feat_cols = [c for c in feat_cols if c in X_full.columns]
        if not feat_cols:
            print(f"  [SKIP] {group_label}: nenhuma feature disponível")
            continue
        X_sub = X_full[feat_cols].copy()
        print(f"  {group_label}: {len(feat_cols)} features")
        for model_name, estimator in _ablation_models.items():
            pipe = build_pipeline(estimator, selector_k=None, n_features=len(feat_cols))
            try:
                fold_df, _ = evaluate_pipeline_groupkfold(
                    pipe, X_sub, y, groups, labels_present, cv
                )
                row = {
                    "study": study_name, "group": group_label,
                    "model": model_name, "n_features": len(feat_cols), "status": "ok",
                }
                row.update(summarize_fold_metrics(fold_df))
                rows.append(row)
            except Exception as e:
                rows.append({
                    "study": study_name, "group": group_label,
                    "model": model_name, "n_features": len(feat_cols),
                    "status": f"error: {e}",
                })
    return pd.DataFrame(rows)


def _best_per_group(abl_df):
    """Retorna o melhor macro_f1_mean por grupo (excluindo erros)."""
    ok = abl_df[abl_df["status"].eq("ok")]
    if ok.empty:
        return pd.DataFrame()
    return (
        ok.sort_values("macro_f1_mean", ascending=False)
        .drop_duplicates(subset=["group"])
        .reset_index(drop=True)
    )


print("Helper run_ablation_study definido.")


Helper run_ablation_study definido.


## 14. Separacao por Origem das Fingerprints

Compara subconjuntos de features agrupados pela **origem no pipeline de extracao**:

| Grupo | Criterio |
|-------|----------|
| `fingerprint_bloco` | Features agregadas/calculadas por bloco de 10 s, apos remover os prefixos `fpband__` e `fprank__` |
| `raw_peaks` | Features brutas de picos (`rp_`, `peak`, `n_original_peaks`, quando presentes no dataset) |
| `global` | Features sem banda explicita |
| `fingerprint_completo` | Todas as features utilizaveis (equivalente ao experimento da secao 7) |

Grupos com zero features sao omitidos automaticamente.


In [37]:
# Deteccao de origem usando o inventario normalizado da secao 6
_origin_map = {
    "fingerprint_bloco": inventory.loc[inventory["origem"].eq("bloco"), "feature"].tolist(),
    "raw_peaks": inventory.loc[inventory["origem"].eq("raw_peaks"), "feature"].tolist(),
    "global": inventory.loc[inventory["banda"].eq("global"), "feature"].tolist(),
    "fingerprint_completo": list(usable_feature_cols),
}
origin_groups = {k: [c for c in v if c in usable_feature_cols] for k, v in _origin_map.items() if v}

print("Grupos de origem detectados:")
for k, v in origin_groups.items():
    pct = len(v) / len(usable_feature_cols) * 100
    print(f"  {k:28s}: {len(v):4d} features ({pct:.1f}%)")

print()
print("Ablation por origem - iniciando...")
origin_ablation = run_ablation_study(origin_groups, X, y, groups, cv, study_name="origem")
save_table(origin_ablation, "ablation_by_origin_fingerprints.csv")


Grupos de origem detectados:
  fingerprint_bloco           :  506 features (67.0%)
  raw_peaks                   :  249 features (33.0%)
  global                      :  257 features (34.0%)
  fingerprint_completo        :  755 features (100.0%)

Ablation por origem - iniciando...
  fingerprint_bloco: 506 features
  raw_peaks: 249 features
  global: 257 features
  fingerprint_completo: 755 features
Tabela salva: C:\dev\python\TCC Ajustado\Dados\pycaret_quadrants_fingerprints_outputs\tables\ablation_by_origin_fingerprints.csv


WindowsPath('C:/dev/python/TCC Ajustado/Dados/pycaret_quadrants_fingerprints_outputs/tables/ablation_by_origin_fingerprints.csv')

In [38]:

_origin_best = _best_per_group(origin_ablation)
if not _origin_best.empty:
    display(Markdown("### Melhor macro-F1 por grupo de origem"))
    display(_origin_best[["group", "model", "n_features", "macro_f1_mean", "macro_f1_std", "balanced_accuracy_mean"]])

    fig_orig = px.bar(
        _origin_best.sort_values("macro_f1_mean"),
        x="macro_f1_mean", y="group", orientation="h",
        color="group",
        error_x="macro_f1_std",
        text=_origin_best.sort_values("macro_f1_mean")["macro_f1_mean"].map(lambda x: f"{x:.3f}"),
        title="Macro-F1 por origem das fingerprints (melhor modelo por grupo)",
    )
    fig_orig.update_layout(
        xaxis_title="Macro-F1 médio", yaxis_title="Grupo de origem", showlegend=False
    )
    save_fig(fig_orig, "ablation_by_origin_fingerprints")

    _ok_orig = origin_ablation[origin_ablation["status"].eq("ok")]
    if len(_ok_orig["model"].unique()) > 1 and len(_ok_orig["group"].unique()) > 1:
        _pivot_orig = _ok_orig.pivot_table(
            index="model", columns="group", values="macro_f1_mean", aggfunc="max"
        )
        fig_heat_orig = px.imshow(
            _pivot_orig, text_auto=".3f", aspect="auto", color_continuous_scale="Blues",
            title="Macro-F1: modelo × grupo de origem",
        )
        fig_heat_orig.update_layout(xaxis_title="Origem", yaxis_title="Modelo")
        save_fig(fig_heat_orig, "ablation_origin_heatmap_fingerprints")


### Melhor macro-F1 por grupo de origem

,group,model,n_features,macro_f1_mean,macro_f1_std,balanced_accuracy_mean
0,global,LogisticRegression_balanced,257,0.451798,0.019492,0.466997
1,fingerprint_completo,LogisticRegression_balanced,755,0.443253,0.027993,0.458887
2,raw_peaks,LogisticRegression_balanced,249,0.431255,0.018220,0.444512
3,fingerprint_bloco,LogisticRegression_balanced,506,0.422125,0.022758,0.438899



## 15. Ablation Study por Banda de Frequência

Isola cada **banda espectral** para quantificar sua contribuição individual ao poder
discriminativo dos quadrantes emocionais.

Bandas avaliadas: `low` · `mid` · `high` · `very_high` · `global` (se presente).

Os grupos são construídos diretamente do `inventory` calculado na seção 6, usando a coluna `banda`.


In [39]:

# Constrói grupos por banda a partir do inventory calculado na seção 6
band_groups = {}
for _band in ["low", "mid", "high", "very_high", "global"]:
    _cols = [
        c for c in inventory.loc[inventory["banda"].eq(_band), "feature"].tolist()
        if c in usable_feature_cols
    ]
    if _cols:
        band_groups[_band] = _cols

print("Grupos por banda:")
for k, v in band_groups.items():
    print(f"  {k:12s}: {len(v):4d} features ({len(v)/len(usable_feature_cols)*100:.1f}%)")

print("\nAblation por banda — iniciando...")
band_ablation = run_ablation_study(band_groups, X, y, groups, cv, study_name="banda")
save_table(band_ablation, "ablation_by_band_fingerprints.csv")


Grupos por banda:
  low         :  123 features (16.3%)
  mid         :  127 features (16.8%)
  high        :  124 features (16.4%)
  very_high   :  124 features (16.4%)
  global      :  257 features (34.0%)

Ablation por banda — iniciando...
  low: 123 features
  mid: 127 features
  high: 124 features
  very_high: 124 features
  global: 257 features
Tabela salva: C:\dev\python\TCC Ajustado\Dados\pycaret_quadrants_fingerprints_outputs\tables\ablation_by_band_fingerprints.csv


WindowsPath('C:/dev/python/TCC Ajustado/Dados/pycaret_quadrants_fingerprints_outputs/tables/ablation_by_band_fingerprints.csv')

In [40]:

_band_best = _best_per_group(band_ablation)
if not _band_best.empty:
    display(Markdown("### Melhor macro-F1 por banda (melhor modelo isolado)"))
    display(_band_best[["group", "model", "n_features", "macro_f1_mean", "macro_f1_std", "balanced_accuracy_mean"]])

    fig_band = px.bar(
        _band_best.sort_values("macro_f1_mean"),
        x="macro_f1_mean", y="group", orientation="h",
        color="group",
        error_x="macro_f1_std",
        text=_band_best.sort_values("macro_f1_mean")["macro_f1_mean"].map(lambda x: f"{x:.3f}"),
        title="Macro-F1 por banda de frequência (melhor modelo isolado)",
    )
    fig_band.update_layout(
        xaxis_title="Macro-F1 médio", yaxis_title="Banda", showlegend=False
    )
    save_fig(fig_band, "ablation_by_band_fingerprints")

    _ok_band = band_ablation[band_ablation["status"].eq("ok")]
    if len(_ok_band["model"].unique()) > 1 and len(_ok_band["group"].unique()) > 1:
        _pivot_band = _ok_band.pivot_table(
            index="model", columns="group", values="macro_f1_mean", aggfunc="max"
        )
        fig_heat_band = px.imshow(
            _pivot_band, text_auto=".3f", aspect="auto", color_continuous_scale="Blues",
            title="Macro-F1: modelo × banda de frequência",
        )
        fig_heat_band.update_layout(xaxis_title="Banda", yaxis_title="Modelo")
        save_fig(fig_heat_band, "ablation_band_heatmap_fingerprints")


### Melhor macro-F1 por banda (melhor modelo isolado)

,group,model,n_features,macro_f1_mean,macro_f1_std,balanced_accuracy_mean
0,global,LogisticRegression_balanced,257,0.451798,0.019492,0.466997
1,mid,LogisticRegression_balanced,127,0.448398,0.020616,0.467096
2,high,LogisticRegression_balanced,124,0.442525,0.019346,0.458235
3,low,LogisticRegression_balanced,123,0.429586,0.020077,0.444811
4,very_high,LogisticRegression_balanced,124,0.418329,0.014087,0.431339



## 16. Ablation Study por Tipo de Feature

Isola cada **tipo de feature** para identificar quais métricas carregam mais informação
emocional nos fingerprints.

| Tipo | Exemplos de features |
|------|----------------------|
| `magnitude` | amplitude dos picos espectrais |
| `rank` | posição relativa dos picos no espectro |
| `pitch_octave` | altura tonal, oitava, semitone, MIDI |
| `counts` | número de eventos, picos por banda |
| `dispersion` | desvio padrão (spread do sinal) |
| `central_tendency` | médias agregadas |
| `frequency` | localização em Hz dos picos |


In [41]:

# Constrói grupos por tipo de feature a partir do inventory (seção 6)
type_groups = {}
for _mtype in ["magnitude", "rank", "pitch_octave", "counts",
               "dispersion", "central_tendency", "frequency", "other"]:
    _cols = [
        c for c in inventory.loc[inventory["metrica"].eq(_mtype), "feature"].tolist()
        if c in usable_feature_cols
    ]
    if _cols:
        type_groups[_mtype] = _cols

print("Grupos por tipo de feature:")
for k, v in type_groups.items():
    print(f"  {k:20s}: {len(v):4d} features ({len(v)/len(usable_feature_cols)*100:.1f}%)")

print("\nAblation por tipo — iniciando...")
type_ablation = run_ablation_study(type_groups, X, y, groups, cv, study_name="tipo")
save_table(type_ablation, "ablation_by_type_fingerprints.csv")


Grupos por tipo de feature:
  magnitude           :  232 features (30.7%)
  rank                :   17 features (2.3%)
  pitch_octave        :  307 features (40.7%)
  counts              :   44 features (5.8%)
  dispersion          :    5 features (0.7%)
  central_tendency    :    9 features (1.2%)
  frequency           :  125 features (16.6%)
  other               :   16 features (2.1%)

Ablation por tipo — iniciando...
  magnitude: 232 features
  rank: 17 features
  pitch_octave: 307 features
  counts: 44 features
  dispersion: 5 features
  central_tendency: 9 features
  frequency: 125 features
  other: 16 features
Tabela salva: C:\dev\python\TCC Ajustado\Dados\pycaret_quadrants_fingerprints_outputs\tables\ablation_by_type_fingerprints.csv


WindowsPath('C:/dev/python/TCC Ajustado/Dados/pycaret_quadrants_fingerprints_outputs/tables/ablation_by_type_fingerprints.csv')

In [42]:

_type_best = _best_per_group(type_ablation)
if not _type_best.empty:
    display(Markdown("### Melhor macro-F1 por tipo de feature (melhor modelo isolado)"))
    display(_type_best[["group", "model", "n_features", "macro_f1_mean", "macro_f1_std", "balanced_accuracy_mean"]])

    fig_type = px.bar(
        _type_best.sort_values("macro_f1_mean"),
        x="macro_f1_mean", y="group", orientation="h",
        color="group",
        error_x="macro_f1_std",
        text=_type_best.sort_values("macro_f1_mean")["macro_f1_mean"].map(lambda x: f"{x:.3f}"),
        title="Macro-F1 por tipo de feature (melhor modelo isolado)",
    )
    fig_type.update_layout(
        xaxis_title="Macro-F1 médio", yaxis_title="Tipo de feature", showlegend=False
    )
    save_fig(fig_type, "ablation_by_type_fingerprints")

    _ok_type = type_ablation[type_ablation["status"].eq("ok")]
    if len(_ok_type["model"].unique()) > 1 and len(_ok_type["group"].unique()) > 1:
        _pivot_type = _ok_type.pivot_table(
            index="model", columns="group", values="macro_f1_mean", aggfunc="max"
        )
        fig_heat_type = px.imshow(
            _pivot_type, text_auto=".3f", aspect="auto", color_continuous_scale="Greens",
            title="Macro-F1: modelo × tipo de feature",
        )
        fig_heat_type.update_layout(xaxis_title="Tipo", yaxis_title="Modelo")
        save_fig(fig_heat_type, "ablation_type_heatmap_fingerprints")


### Melhor macro-F1 por tipo de feature (melhor modelo isolado)

,group,model,n_features,macro_f1_mean,macro_f1_std,balanced_accuracy_mean
0,magnitude,LogisticRegression_balanced,232,0.448457,0.015043,0.467291
1,central_tendency,LogisticRegression_balanced,9,0.377134,0.034313,0.402104
2,frequency,LogisticRegression_balanced,125,0.360054,0.016002,0.383774
3,counts,LogisticRegression_balanced,44,0.354019,0.014106,0.357400
4,pitch_octave,LogisticRegression_balanced,307,0.350229,0.018448,0.364608
5,other,LogisticRegression_balanced,16,0.312154,0.021497,0.342290
6,rank,LogisticRegression_balanced,17,0.301324,0.023359,0.312091
7,dispersion,LogisticRegression_balanced,5,0.282707,0.010261,0.293789



## 17. StratifiedGroupKFold — Protocolo Secundário Abrangente

Reavalia **todos os modelos** com `StratifiedGroupKFold` — preserva a proporção de classes
por fold enquanto mantém grupos de `song_id` intactos — e compara com os resultados do
protocolo `GroupKFold` principal.

**Objetivo:** confirmar a estabilidade das estimativas e verificar se a estratificação
altera o ranking de modelos ou inflaciona/deflaciona as métricas.


In [43]:

if StratifiedGroupKFold is not None:
    sgkf = StratifiedGroupKFold(n_splits=cv_splits)
    sgkf_rows, sgkf_folds = [], []

    for model_name, estimator in tqdm(build_candidate_models().items(), desc="StratifiedGroupKFold"):
        for sel_k in [None] + [k for k in SELECTOR_K_VALUES if k < len(usable_feature_cols)]:
            sel_name = "none" if sel_k is None else f"SelectKBest(k={sel_k})"
            pipe = build_pipeline(estimator, selector_k=sel_k, n_features=len(usable_feature_cols))
            try:
                fold_df, _ = evaluate_pipeline_groupkfold(
                    pipe, X, y, groups, labels_present, sgkf
                )
                fold_df.insert(0, "selector", sel_name)
                fold_df.insert(0, "model",    model_name)
                sgkf_folds.append(fold_df)
                row = {
                    "model": model_name, "selector": sel_name,
                    "feature_set": "fingerprints",
                    "cv": f"StratifiedGroupKFold({cv_splits}) por song_id",
                    "n_features_effective": (
                        len(usable_feature_cols) if sel_k is None
                        else min(sel_k, len(usable_feature_cols))
                    ),
                    "status": "ok",
                }
                row.update(summarize_fold_metrics(fold_df))
                sgkf_rows.append(row)
            except Exception as e:
                sgkf_rows.append({
                    "model": model_name, "selector": sel_name,
                    "feature_set": "fingerprints",
                    "cv": f"StratifiedGroupKFold({cv_splits})", "status": f"error: {e}",
                })

    sgkf_results = pd.DataFrame(sgkf_rows).sort_values(
        ["status", "macro_f1_mean"], ascending=[False, False], na_position="last"
    )
    save_table(sgkf_results, "classification_stratifiedgroupkfold_results_fingerprints.csv")
    if sgkf_folds:
        save_table(
            pd.concat(sgkf_folds, ignore_index=True),
            "classification_stratifiedgroupkfold_fold_metrics_fingerprints.csv",
        )

    ok_sgkf = sgkf_results[sgkf_results["status"].eq("ok")]

    # Comparação direta GroupKFold vs StratifiedGroupKFold para cada modelo
    _gkf_best_m = (
        ok_results[~ok_results["model"].str.startswith("Dummy")]
        .sort_values("macro_f1_mean", ascending=False)
        .drop_duplicates(subset=["model"])
        .set_index("model")[["macro_f1_mean", "macro_f1_std"]]
        .rename(columns={"macro_f1_mean": "gkf_f1_mean", "macro_f1_std": "gkf_f1_std"})
    )
    _sgkf_best_m = (
        ok_sgkf[~ok_sgkf["model"].str.startswith("Dummy")]
        .sort_values("macro_f1_mean", ascending=False)
        .drop_duplicates(subset=["model"])
        .set_index("model")[["macro_f1_mean", "macro_f1_std"]]
        .rename(columns={"macro_f1_mean": "sgkf_f1_mean", "macro_f1_std": "sgkf_f1_std"})
    )
    cv_comp_all = _gkf_best_m.join(_sgkf_best_m, how="inner").reset_index()
    cv_comp_all["delta_sgkf_minus_gkf"] = cv_comp_all["sgkf_f1_mean"] - cv_comp_all["gkf_f1_mean"]
    cv_comp_all = cv_comp_all.sort_values("sgkf_f1_mean", ascending=False)
    save_table(cv_comp_all, "groupkfold_vs_stratified_all_models_fingerprints.csv")

    display(Markdown("### GroupKFold vs StratifiedGroupKFold — todos os modelos"))
    display(cv_comp_all)

    fig_cv_comp = px.scatter(
        cv_comp_all,
        x="gkf_f1_mean", y="sgkf_f1_mean",
        text="model",
        title="GroupKFold vs StratifiedGroupKFold — Macro-F1 por modelo",
        labels={"gkf_f1_mean": "GroupKFold Macro-F1", "sgkf_f1_mean": "StratifiedGroupKFold Macro-F1"},
    )
    _axis_min = cv_comp_all[["gkf_f1_mean", "sgkf_f1_mean"]].min().min() - 0.02
    _axis_max = cv_comp_all[["gkf_f1_mean", "sgkf_f1_mean"]].max().max() + 0.02
    fig_cv_comp.add_shape(
        type="line", x0=_axis_min, y0=_axis_min, x1=_axis_max, y1=_axis_max,
        line=dict(dash="dash", color="gray"),
    )
    fig_cv_comp.update_traces(textposition="top center")
    save_fig(fig_cv_comp, "groupkfold_vs_stratified_scatter_fingerprints")
else:
    sgkf_results, ok_sgkf = pd.DataFrame(), pd.DataFrame()
    print("StratifiedGroupKFold não disponível nesta versão do scikit-learn.")


StratifiedGroupKFold: 100%|██████████| 9/9 [08:44<00:00, 58.23s/it]

Tabela salva: C:\dev\python\TCC Ajustado\Dados\pycaret_quadrants_fingerprints_outputs\tables\classification_stratifiedgroupkfold_results_fingerprints.csv
Tabela salva: C:\dev\python\TCC Ajustado\Dados\pycaret_quadrants_fingerprints_outputs\tables\classification_stratifiedgroupkfold_fold_metrics_fingerprints.csv
Tabela salva: C:\dev\python\TCC Ajustado\Dados\pycaret_quadrants_fingerprints_outputs\tables\groupkfold_vs_stratified_all_models_fingerprints.csv


### GroupKFold vs StratifiedGroupKFold — todos os modelos

,model,gkf_f1_mean,gkf_f1_std,sgkf_f1_mean,sgkf_f1_std,delta_sgkf_minus_gkf
0,LogisticRegression_balanced,0.468302,0.020579,0.466724,0.026831,-0.001578
1,RidgeClassifier_balanced,0.456543,0.023758,0.460268,0.030199,0.003726
2,GaussianNB,0.450541,0.020778,0.459206,0.035830,0.008665
3,RandomForest_balanced,0.433171,0.023985,0.428767,0.022059,-0.004403
4,ExtraTrees_balanced,0.426332,0.024568,0.424135,0.026160,-0.002198
5,KNeighbors,0.413307,0.017443,0.409925,0.034888,-0.003381
6,DecisionTree_balanced,0.367764,0.013056,0.375113,0.017038,0.007350


## 18. Melhor Tecnica de Fingerprint: Rank vs Band-Rank e Bloco vs Raw-Peaks

Esta secao cruza as duas perguntas finais do experimento:

- qual tecnica teve melhor desempenho: `rank` ou `band-rank`;
- de onde veio o melhor subconjunto: features de `bloco` ou `raw-peaks`.

Os raw-peaks do `band-rank` sao lidos de `fingerprint_band_rank_raw_peaks.parquet`, agregados por bloco e comparados junto com os demais grupos. O mesmo protocolo de ablation (`GroupKFold` por `song_id`, Macro-F1 como metrica principal) e aplicado aos grupos `tecnica x origem`.


In [44]:
# Comparacao cruzada tecnica x origem
tech_origin_groups = {}
for _tech in ["rank", "band_rank"]:
    for _origin in ["bloco", "raw_peaks"]:
        _cols = inventory.loc[
            inventory["tecnica"].eq(_tech) & inventory["origem"].eq(_origin),
            "feature",
        ].tolist()
        _cols = [c for c in _cols if c in usable_feature_cols]
        if _cols:
            tech_origin_groups[f"{_tech}__{_origin}"] = _cols

print("Grupos tecnica x origem detectados:")
for k, v in tech_origin_groups.items():
    pct = len(v) / len(usable_feature_cols) * 100
    print(f"  {k:24s}: {len(v):4d} features ({pct:.1f}%)")

if tech_origin_groups:
    print("\nAblation tecnica x origem - iniciando...")
    tech_origin_ablation = run_ablation_study(
        tech_origin_groups, X, y, groups, cv, study_name="tecnica_origem"
    )
    save_table(tech_origin_ablation, "ablation_by_technique_origin_fingerprints.csv")

    _tech_origin_best = _best_per_group(tech_origin_ablation)
    if _tech_origin_best.empty:
        fingerprint_final_winner_df = pd.DataFrame()
        print("[AVISO] A comparacao tecnica x origem nao teve modelos validos.")
    else:
        _tech_origin_best = _tech_origin_best.sort_values(
            ["macro_f1_mean", "balanced_accuracy_mean", "accuracy_mean"],
            ascending=False,
            na_position="last",
        ).reset_index(drop=True)

        _split = _tech_origin_best["group"].str.split("__", n=1, expand=True)
        _tech_origin_best["tecnica"] = _split[0]
        _tech_origin_best["origem"] = _split[1]
        _tech_origin_best["tecnica_label"] = _tech_origin_best["tecnica"].map({
            "rank": "Rank",
            "band_rank": "Band-rank",
        }).fillna(_tech_origin_best["tecnica"])
        _tech_origin_best["origem_label"] = _tech_origin_best["origem"].map({
            "bloco": "Bloco",
            "raw_peaks": "Raw-peaks",
        }).fillna(_tech_origin_best["origem"])

        save_table(_tech_origin_best, "best_by_technique_origin_fingerprints.csv")

        display(Markdown("### Ranking por tecnica x origem"))
        display(_tech_origin_best[[
            "tecnica_label", "origem_label", "model", "n_features",
            "macro_f1_mean", "macro_f1_std", "balanced_accuracy_mean",
        ]])

        _winner = _tech_origin_best.iloc[0]
        fingerprint_final_winner_df = pd.DataFrame([{
            "tecnica_vencedora": _winner["tecnica_label"],
            "origem_vencedora": _winner["origem_label"],
            "grupo": _winner["group"],
            "modelo": _winner["model"],
            "n_features": int(_winner["n_features"]),
            "macro_f1_mean": float(_winner["macro_f1_mean"]),
            "macro_f1_std": float(_winner.get("macro_f1_std", np.nan)),
            "balanced_accuracy_mean": float(_winner.get("balanced_accuracy_mean", np.nan)),
        }])
        save_table(fingerprint_final_winner_df, "fingerprint_best_technique_origin_winner.csv")

        display(Markdown(
            "### Conclusao - tecnica de fingerprint vencedora\n"
            f"O melhor desempenho entre `rank` e `band-rank`, cruzando `bloco` e `raw-peaks`, "
            f"foi **{_winner['tecnica_label']} / {_winner['origem_label']}** "
            f"com **Macro-F1 medio = {_winner['macro_f1_mean']:.4f} +/- {_winner.get('macro_f1_std', np.nan):.4f}** "
            f"usando **{_winner['model']}** e **{int(_winner['n_features'])} features**."
        ))

        fig_tech_origin = px.bar(
            _tech_origin_best.sort_values("macro_f1_mean"),
            x="macro_f1_mean", y="group", orientation="h",
            color="tecnica_label",
            error_x="macro_f1_std",
            text=_tech_origin_best.sort_values("macro_f1_mean")["macro_f1_mean"].map(lambda x: f"{x:.3f}"),
            title="Macro-F1 por tecnica de fingerprint x origem",
        )
        fig_tech_origin.update_layout(
            xaxis_title="Macro-F1 medio",
            yaxis_title="Tecnica x origem",
            legend_title="Tecnica",
        )
        save_fig(fig_tech_origin, "ablation_by_technique_origin_fingerprints")
else:
    tech_origin_ablation = pd.DataFrame()
    fingerprint_final_winner_df = pd.DataFrame()
    print("[AVISO] Nenhum grupo tecnica x origem disponivel para comparacao final.")


Grupos tecnica x origem detectados:
  rank__bloco             :  204 features (27.0%)
  rank__raw_peaks         :    3 features (0.4%)
  band_rank__bloco        :  302 features (40.0%)
  band_rank__raw_peaks    :  246 features (32.6%)

Ablation tecnica x origem - iniciando...
  rank__bloco: 204 features
  rank__raw_peaks: 3 features
  band_rank__bloco: 302 features
  band_rank__raw_peaks: 246 features
Tabela salva: C:\dev\python\TCC Ajustado\Dados\pycaret_quadrants_fingerprints_outputs\tables\ablation_by_technique_origin_fingerprints.csv
Tabela salva: C:\dev\python\TCC Ajustado\Dados\pycaret_quadrants_fingerprints_outputs\tables\best_by_technique_origin_fingerprints.csv


### Ranking por tecnica x origem

,tecnica_label,origem_label,model,n_features,macro_f1_mean,macro_f1_std,balanced_accuracy_mean
0,Band-rank,Raw-peaks,LogisticRegression_balanced,246,0.431013,0.021021,0.444835
1,Band-rank,Bloco,LogisticRegression_balanced,302,0.418228,0.015593,0.437494
2,Rank,Bloco,LogisticRegression_balanced,204,0.400085,0.020419,0.418813
3,Rank,Raw-peaks,ExtraTrees_balanced,3,0.322668,0.017240,0.335303


Tabela salva: C:\dev\python\TCC Ajustado\Dados\pycaret_quadrants_fingerprints_outputs\tables\fingerprint_best_technique_origin_winner.csv


### Conclusao - tecnica de fingerprint vencedora
O melhor desempenho entre `rank` e `band-rank`, cruzando `bloco` e `raw-peaks`, foi **Band-rank / Raw-peaks** com **Macro-F1 medio = 0.4310 +/- 0.0210** usando **LogisticRegression_balanced** e **246 features**.

In [45]:

# ===========================
# Sumário geral de todos os experimentos
# ===========================

def _best_row_summary(df, label, cv_label):
    if df.empty or "macro_f1_mean" not in df.columns:
        return None
    _ok = df[df["status"].eq("ok")] if "status" in df.columns else df
    _nd = _ok[~_ok["model"].str.startswith("Dummy")] if "model" in _ok.columns else _ok
    if _nd.empty:
        return None
    b = _nd.sort_values("macro_f1_mean", ascending=False).iloc[0]
    return {
        "Experimento":    label,
        "CV":             cv_label,
        "Modelo":         b.get("model", ""),
        "Seletor":        b.get("selector", ""),
        "Macro-F1 médio": round(float(b.get("macro_f1_mean", np.nan)), 4),
        "Macro-F1 std":   round(float(b.get("macro_f1_std",  np.nan)), 4),
        "Balanced Acc":   round(float(b.get("balanced_accuracy_mean", np.nan)), 4),
        "N° features":    int(b.get("n_features_effective", b.get("n_features", 0))),
    }


_gkf_cv  = f"GroupKFold({cv_splits})"
_sgkf_cv = f"StratifiedGroupKFold({cv_splits})"

_summary_entries = [
    _best_row_summary(ok_results, "Fingerprints", _gkf_cv),
    _best_row_summary(globals().get("corr_filter_results", pd.DataFrame()), "Fingerprints sem redundância", _gkf_cv),
    _best_row_summary(globals().get("ok_comb",  pd.DataFrame()), "openSMILE + Fingerprints", _gkf_cv),
    _best_row_summary(globals().get("fusion_variant_results", pd.DataFrame()), "Melhor fusão controlada", _gkf_cv),
    _best_row_summary(globals().get("ok_sgkf",  pd.DataFrame()), "Fingerprints (SGKF)",      _sgkf_cv),
]

if not opensmile_results.empty:
    _os_ok3 = (opensmile_results if "status" not in opensmile_results.columns
               else opensmile_results[opensmile_results["status"].eq("ok")])
    _summary_entries.insert(1, _best_row_summary(_os_ok3, "openSMILE", _gkf_cv))

for _abl_df, _abl_label in [
    (globals().get("origin_ablation", pd.DataFrame()), "Ablation — melhor origem"),
    (globals().get("band_ablation",   pd.DataFrame()), "Ablation — melhor banda"),
    (globals().get("type_ablation",   pd.DataFrame()), "Ablation — melhor tipo"),
    (globals().get("tech_origin_ablation", pd.DataFrame()), "Ablation - melhor tecnica/origem"),
]:
    if not _abl_df.empty and "macro_f1_mean" in _abl_df.columns:
        _ok_a = _abl_df[_abl_df["status"].eq("ok")]
        if not _ok_a.empty:
            _ba = _ok_a.sort_values("macro_f1_mean", ascending=False).iloc[0]
            _summary_entries.append({
                "Experimento":    f"{_abl_label} ({_ba['group']})",
                "CV":             _gkf_cv,
                "Modelo":         _ba["model"],
                "Seletor":        "none",
                "Macro-F1 médio": round(float(_ba["macro_f1_mean"]), 4),
                "Macro-F1 std":   round(float(_ba.get("macro_f1_std", np.nan)), 4),
                "Balanced Acc":   round(float(_ba.get("balanced_accuracy_mean", np.nan)), 4),
                "N° features":    int(_ba["n_features"]),
            })

summary_all_df = pd.DataFrame([e for e in _summary_entries if e is not None])
save_table(summary_all_df, "tcc_summary_all_experiments.csv")

display(Markdown("## Sumário Geral — Todos os Experimentos"))
display(summary_all_df)

if len(summary_all_df) > 1 and "Macro-F1 médio" in summary_all_df.columns:
    _sa_sorted = summary_all_df.sort_values("Macro-F1 médio")
    fig_all = px.bar(
        _sa_sorted,
        x="Macro-F1 médio", y="Experimento", orientation="h",
        error_x="Macro-F1 std",
        color="Macro-F1 médio",
        color_continuous_scale="Viridis",
        text=_sa_sorted["Macro-F1 médio"].map(lambda x: f"{x:.3f}"),
        title="Macro-F1 — Visão geral de todos os experimentos (TCC)",
    )
    fig_all.update_layout(
        xaxis_title="Macro-F1 médio (GroupKFold por song_id)",
        yaxis_title="Experimento",
        showlegend=False,
    )
    save_fig(fig_all, "summary_all_experiments")


if not globals().get("fingerprint_final_winner_df", pd.DataFrame()).empty:
    _fw = fingerprint_final_winner_df.iloc[0]
    display(Markdown(
        "## Conclusao Final - Tecnica de Fingerprint\n"
        f"A melhor combinacao final foi **{_fw['tecnica_vencedora']} / {_fw['origem_vencedora']}**, "
        f"com **Macro-F1 medio = {_fw['macro_f1_mean']:.4f}** "
        f"(**{_fw['modelo']}**, {int(_fw['n_features'])} features)."
    ))
    display(fingerprint_final_winner_df)


Tabela salva: C:\dev\python\TCC Ajustado\Dados\pycaret_quadrants_fingerprints_outputs\tables\tcc_summary_all_experiments.csv


## Sumário Geral — Todos os Experimentos

,Experimento,CV,Modelo,Seletor,Macro-F1 médio,Macro-F1 std,Balanced Acc,N° features
0,Fingerprints,GroupKFold(5),LogisticRegression_balanced,SelectKBest(k=200),0.4683,0.0206,0.4864,200
1,Fingerprints (SGKF),StratifiedGroupKFold(5),LogisticRegression_balanced,SelectKBest(k=200),0.4667,0.0268,0.4833,200
2,Ablation — melhor origem (global),GroupKFold(5),LogisticRegression_balanced,none,0.4518,0.0195,0.4670,257
3,Ablation — melhor banda (global),GroupKFold(5),LogisticRegression_balanced,none,0.4518,0.0195,0.4670,257
4,Ablation — melhor tipo (magnitude),GroupKFold(5),LogisticRegression_balanced,none,0.4485,0.0150,0.4673,232
5,Ablation - melhor tecnica/origem (band_rank__r...,GroupKFold(5),LogisticRegression_balanced,none,0.4310,0.0210,0.4448,246


Resorting to unclean kill browser.
Resorting to unclean kill browser.


[AVISO] SVG não exportado: Couldn't close or kill browser subprocess


## Conclusao Final - Tecnica de Fingerprint
A melhor combinacao final foi **Band-rank / Raw-peaks**, com **Macro-F1 medio = 0.4310** (**LogisticRegression_balanced**, 246 features).

,tecnica_vencedora,origem_vencedora,grupo,modelo,n_features,macro_f1_mean,macro_f1_std,balanced_accuracy_mean
0,Band-rank,Raw-peaks,band_rank__raw_peaks,LogisticRegression_balanced,246,0.431013,0.021021,0.444835
